# Physics-Informed Neural Networks for Data-Efficient 4D-STEM Strain Reconstruction

**Goal**: Reconstruct strain maps (e_xx, e_yy, e_xy, and the derived in-plane angle θ) of the
IV–VI high-entropy thermoelectric PbGeSnSe₁.₅Te₁.₅ (domain-structured, ferroelastic; Liu et al.,
*JACS* 2024, 146, 12620–12635) from sparse subsets (**1%–75%**) of the 4D-STEM probe positions,
and quantify when a linear-elasticity prior actually helps.

### Method

1. **SIREN backbone** (Sitzmann et al., 2020): sine-activated coordinate network (ω₀ = 30) with skip
   connections, mapping normalised (x, y) to the four field components.
2. **Physics constraints as soft losses**: plane-stress elastic equilibrium (∇·σ = 0) and
   Saint-Venant compatibility, evaluated by reverse-mode automatic differentiation at random
   collocation points.
3. **Scale-normalised physics weighting**: each residual is divided by a detached running (EMA)
   estimate of its own magnitude so physics gradients stay commensurate with the O(1) data-MSE term;
   the overall physics weight follows an exponential ramp λ(t) = λ_max·(1 − e^(−t/τ)).
4. **Residual-based adaptive refinement (RAR)**: the collocation pool is grown where the equilibrium
   residual is largest. Note this refines *physics collocation points*; it does not add measured data.
5. **Evaluation** at sampling fractions 1–75% against compressed sensing (wavelet ISTA),
   Gaussian-process regression, and a data-only SIREN ablation, plus Bayesian epistemic uncertainty
   (MC dropout and mean-field VI, Section 10).

### Headline results (seed-fixed run; see Sections 8–8d)

- R²(e_xx) rises from ≈0.44 at 1% sampling to ≈0.80 at 10% and saturates by ≈25% (0.84 vs 0.86 at
  75%): **10–25% of probe positions capture essentially all recoverable structure**. The residual
  gap to R² = 1 is dominated by pixel-scale speckle, which is largely measurement noise (~3% of the
  field variance).
- At 10% sampling the physics-informed model reduces MAE(e_xx) by ≈26% vs compressed sensing and
  ≈22% vs GP regression.
- The elasticity prior behaves as a **sparse-data regulariser**: it improves accuracy at 1%
  sampling (≈10% lower MAE), is roughly neutral at 5%, and biases the fit as data grows (see
  Conclusions).

### References
- [SIREN: Implicit Neural Representations with Periodic Activation Functions](https://arxiv.org/abs/2006.09661)
- [Curriculum-Enhanced Adaptive Sampling for PINNs](https://www.mdpi.com/2227-7390/13/24/3996)
- [Annealed Adaptive Importance Sampling (AAIS)](https://arxiv.org/abs/2405.03433)
- [R-PINN: Recovery-type Error Estimators](https://arxiv.org/html/2506.10243)
- [Failure-Informed Adaptive Sampling](https://epubs.siam.org/doi/10.1137/22M1527763)
- [Investigating Guiding Information for Adaptive Sampling](https://arxiv.org/html/2404.12282v1)


## ⚡ Colab GPU version

This is a duplicate of `pinns-strain-sota-adaptive-2.ipynb` adapted to run on Colab GPUs
(Runtime ▸ Change runtime type ▸ **GPU**). Expected runtime on a T4: roughly 20–40 min end-to-end.

**Environment changes** (marked `[COLAB]`):
1. CUDA-first device selection.
2. Setup cells below: dependency check + data loading (local → Google Drive → manual upload).
3. matplotlib ≥ 3.9 API compatibility in the boxplot cell.

**Verified fixes** (marked `[FIX]`) — the original run produced R² ≈ 0.04–0.19 at *every*
sampling fraction because (a) ω₀ was hard-coded to 1.0 inside `create_pinn_model` (the SIREN
was nearly linear and underfit) and (b) the physics residuals — O(1e2)–O(1e4) on standardised
fields over unit coordinates — crushed the O(1) data term and drove the network to a
near-constant field (early-stopping at ~350/5000 epochs with val loss ≈ 0.9 ≈ data variance):
1. `omega_0 = 30.0`, and it is now actually passed through to the model.
2. `normalize_residuals = True`: each physics residual is divided by a detached running (EMA)
   estimate of its own magnitude before weighting — with the EMA **frozen after a 200-step
   warmup** (a live EMA eventually collapses long runs to a flat field).
3. Section-10 UQ repairs: `train_bayes_model` ramps λ_phys, MC dropout uses p=0.05 on hidden
   layers only, and the MFVI prior/noise are matched to the SIREN weight scale.

Verified at 10% sampling (data-only R²(e_xx) = 0.82; physics-on with fixes = 0.81 vs 0.04 before).
Set `omega_0: 1.0` and `normalize_residuals: False` in the config to reproduce the original run.


In [ ]:
# ============================================================================
# COLAB SETUP 1/2: dependencies & GPU check   [COLAB]
# ============================================================================
import importlib, subprocess, sys

def _ensure(module, pip_name=None):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or module])

_ensure("pywt", "PyWavelets")     # compressed-sensing baseline (Section 8d)
_ensure("sklearn", "scikit-learn")
_ensure("tqdm")
_ensure("pandas")

import torch
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠ No GPU detected — Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ GPU")


In [ ]:
# ============================================================================
# COLAB SETUP 2/2: strain data (~1.7 MB total)   [COLAB]
# Looks for strain_exx.npy / strain_exy.npy / strain_eyy.npy in:
#   1. ./data  (already present, e.g. running locally)
#   2. Google Drive (set DRIVE_DATA_DIR below, approve the mount prompt)
#   3. manual upload prompt
# ============================================================================
from pathlib import Path

DATA_FILES = ["strain_exx.npy", "strain_exy.npy", "strain_eyy.npy"]
DRIVE_DATA_DIR = "/content/drive/MyDrive/pinns-4dstem/data"   # <-- adjust to your Drive layout

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

def _have_all():
    return all((data_dir / f).exists() for f in DATA_FILES)

if not _have_all():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        src = Path(DRIVE_DATA_DIR)
        if all((src / f).exists() for f in DATA_FILES):
            import shutil
            for f in DATA_FILES:
                shutil.copy(src / f, data_dir / f)
            print(f"✓ Copied data from {src}")
        else:
            print(f"⚠ Data not found in {src} — falling back to upload")
    except Exception as exc:
        print(f"(Drive not used: {exc})")

if not _have_all():
    from google.colab import files
    print("Upload strain_exx.npy, strain_exy.npy, strain_eyy.npy:")
    uploaded = files.upload()
    for name, blob in uploaded.items():
        (data_dir / Path(name).name).write_bytes(blob)

assert _have_all(), f"Missing data files in {data_dir.resolve()}: {DATA_FILES}"
print("✓ Data ready:", sorted(p.name for p in data_dir.glob("strain_*.npy")))


Mounted at /content/drive
⚠ Data not found in /content/drive/MyDrive/pinns-4dstem/data — falling back to upload
Upload strain_exx.npy, strain_exy.npy, strain_eyy.npy:


: 

In [ ]:
# ============================================================================
# SECTION 1: Environment Setup & Configuration
# ============================================================================
import sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import csv
import time
from tqdm import trange
from collections import defaultdict
import pandas as pd

# Versions
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")

# Device selection (prefer CUDA on Colab, then Apple MPS, then CPU)  [COLAB]
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✓ Using CUDA: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Using Apple Metal Performance Shaders (MPS)")
else:
    device = torch.device("cpu")
    print("⚠ Using CPU (slow!) — Colab: Runtime ▸ Change runtime type ▸ GPU")

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (12, 8),
    "image.cmap": "RdBu",
    "axes.grid": False,
    "font.size": 14,
    "font.family": "sans-serif",
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "figure.titlesize": 15,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

print("\n" + "="*70)
print("CONFIGURATION: STATE-OF-THE-ART PINN FOR STRAIN RECONSTRUCTION")
print("="*70)

In [ ]:
# ============================================================================
# Comprehensive Configuration Dictionary
# ============================================================================
print("\n" + "="*70)
print("CONFIGURATION: PHYSICS-INFORMED PINN FOR STRAIN RECONSTRUCTION")
print("="*70)

config = {
    # --- DATA & GRID ---
    "sampling_fractions": [0.01, 0.05, 0.10, 0.25, 0.50, 0.75],
    "full_data_training": False,

    # --- SPECIMEN: PbGeSnSe1.5Te1.5, IV-VI high-entropy thermoelectric (worked example) ---
    # Isotropic linear-elastic constants used to close elastic equilibrium.
    "specimen": "PbGeSnSe1.5Te1.5 (IV-VI high-entropy thermoelectric)",
    "E": 150e9,                 # Young's modulus (Pa); cancels under scale-normalised residuals (Requirement 3) -- only nu enters training
    "nu": 0.27,                 # Poisson's ratio
    "px_um": 0.017872741,       # real-space pixel size (micrometre/pixel) from metadata

    # --- PHYSICS LOSS (autodiff @ collocation points) ---
    "n_collocation": 2000,      # collocation points resampled each step (Eq. for L_phys)
    "alpha_eq": 0.1,            # weight on elastic-equilibrium residual
    "alpha_co": 0.1,            # weight on Saint-Venant compatibility residual
    "w_data": 1.0,              # data-term weight (lambda_data)
    "lambda_phys_max": 1.0,     # asymptotic physics weight (lambda_phys^max)
    "ramp_tau": 500.0,          # ramp time-constant tau (epochs): lambda_phys(t)=max*(1-e^{-t/tau})

    "normalize_residuals": True,  # [FIX] scale-free physics weighting; False reproduces the original run
                                  #       (raw residuals are O(1e2)-O(1e4) on standardised fields over unit
                                  #        coords and crush the O(1) data term → flat, useless reconstructions)
    "phys_norm_warmup": 200,      # [FIX] freeze the residual-scale EMA after this many steps

    # --- ARCHITECTURE (SIREN backbone) ---
    "architecture": "siren",
    "hidden_dim": 128,
    "num_layers": 6,
    "skip_connections": True,
    "omega_0": 30.0,           # [FIX] was 1.0 — SIREN needs ω₀≈30 (Sitzmann et al. 2020); ω₀=1 underfits badly

    # --- TRAINING ---
    "epochs": 5000,
    "lr": 1e-3,
    "lr_decay_factor": 0.95,
    "lr_decay_interval": 500,
    "early_stop": True,
    "patience": 300,            # patience on held-out VALIDATION data loss
    "min_delta": 1e-6,
    "val_fraction": 0.10,       # fraction of sampled points held out for validation/early-stop

    # --- ADAPTIVE COLLOCATION REFINEMENT (RAR) ---
    "enable_rar": True,
    "rar_trigger_epoch": 2000,
    "rar_interval": 500,
    "rar_num_new_points_pct": 0.10,   # add 10% of n_collocation high-residual points
    "rar_max_pool_mult": 5,           # cap persistent pool at 5x n_collocation

    # --- OUTPUTS ---
    "out_dir": "outputs/sota_adaptive-2-colab",
    "save_checkpoints": False,
    "verbose": True,
    "compute_full_domain_error": True,
}

out_dir = Path(config["out_dir"]).resolve()
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Specimen           : {config['specimen']}  (E={config['E']:.3g} Pa, nu={config['nu']})")
print(f"Pixel size         : {config['px_um']*1000:.3f} nm/pixel")
print(f"Sampling fractions : {config['sampling_fractions']}")
print(f"Architecture       : {config['architecture'].upper()} "
      f"({config['num_layers']} layers x {config['hidden_dim']}, omega_0={config['omega_0']})")
print(f"Physics            : autodiff equilibrium + Saint-Venant compatibility @ "
      f"{config['n_collocation']} collocation pts; ramp tau={config['ramp_tau']:.0f}")
print(f"RAR                : {config['enable_rar']} (from epoch {config['rar_trigger_epoch']}, "
      f"every {config['rar_interval']}, +{config['rar_num_new_points_pct']*100:.0f}%)")
print(f"Output dir         : {out_dir}")


In [ ]:
# ============================================================================
# SECTION 2: Load & Visualize 4D-STEM Strain Data (SEPARATE FILES)
# ============================================================================

# Load separate strain component files
root = Path.cwd()
data_candidates = [
    (root / 'data' / 'strain_exx.npy', root / 'data' / 'strain_exy.npy', root / 'data' / 'strain_eyy.npy'),
    (root / 'strain_exx.npy', root / 'strain_exy.npy', root / 'strain_eyy.npy'),
]

data_paths = None
for paths in data_candidates:
    if all(p.exists() for p in paths):
        data_paths = paths
        break

if data_paths is None:
    raise FileNotFoundError("Could not find strain component files (strain_exx.npy, strain_exy.npy, strain_eyy.npy)")

exx_path, exy_path, eyy_path = data_paths
print(f"✓ Loading strain components from:")
print(f"   - {exx_path}")
print(f"   - {exy_path}")
print(f"   - {eyy_path}")

e_xx = np.load(exx_path)
e_xy = np.load(exy_path)
e_yy = np.load(eyy_path)

# Auto-compute derived fields from components
mask = np.isfinite(e_xx).astype(np.float32)
theta = np.arctan2(e_xy, (e_xx - e_yy) / 2.0 + 1e-10)
error = np.sqrt(e_xx**2 + e_yy**2 + 2*e_xy**2)

H, W = e_xx.shape
print(f"✓ Grid dimensions: {H} × {W} = {H*W:,} points")
print(f"✓ Valid masked region: {mask.sum():,} / {H*W:,} points ({100*mask.sum()/(H*W):.1f}%)")

# Compute normalization statistics on valid entries
valid = mask.astype(bool)
scalers = {}
for name, arr in {
    'e_xx': e_xx,
    'e_yy': e_yy,
    'e_xy': e_xy,
    'theta': theta,
}.items():
    vals = arr[valid]
    mean_val = float(np.nanmean(vals))
    std_val = float(np.nanstd(vals)) + 1e-8
    scalers[name] = {"mean": mean_val, "std": std_val}
    print(f"   {name:8s}: μ={mean_val:+.4e}, σ={std_val:.4e}")

# Scaling functions
def scale(arr, name):
    s = scalers[name]
    return (arr - s["mean"]) / s["std"]

def unscale(arr_tensor, name):
    s = scalers[name]
    if isinstance(arr_tensor, torch.Tensor):
        return arr_tensor * s["std"] + s["mean"]
    return arr_tensor * s["std"] + s["mean"]

# Create spatial grid
x_lin = torch.linspace(0, 1, W, dtype=torch.float32)
y_lin = torch.linspace(0, 1, H, dtype=torch.float32)
X, Y = torch.meshgrid(x_lin, y_lin, indexing='xy')

# Flatten and move to device
x_flat = X.flatten().to(device)
y_flat = Y.flatten().to(device)

# Prepare tensors with scaling
e_xx_scaled = torch.tensor(scale(e_xx, 'e_xx').flatten(), dtype=torch.float32, device=device)
e_yy_scaled = torch.tensor(scale(e_yy, 'e_yy').flatten(), dtype=torch.float32, device=device)
e_xy_scaled = torch.tensor(scale(e_xy, 'e_xy').flatten(), dtype=torch.float32, device=device)
theta_scaled = torch.tensor(scale(theta, 'theta').flatten(), dtype=torch.float32, device=device)
mask_flat = torch.tensor(mask.flatten(), dtype=torch.float32, device=device)

print(f"✓ Tensors prepared and moved to device: {device}")

In [ ]:
# Visualize raw strain data
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('4D-STEM Strain Components (Raw Data)', fontsize=14, fontweight='bold')

components = [
    (e_xx, 'e_xx', 0),
    (e_yy, 'e_yy', 1),
    (e_xy, 'e_xy', 2),
    (theta, 'θ (rotation)', 3),
    (mask, 'Valid Mask', 4),
    (error, 'Measurement Error', 5),
]

for arr, label, idx in components:
    ax = axes.flat[idx]
    
    # Replace NaN/Inf with valid values for visualization
    arr_display = np.copy(arr).astype(float)
    arr_display[~np.isfinite(arr_display)] = 0.0
    
    # Compute vmin/vmax from valid data only
    # Use only finite values to avoid NaN/Inf issues
    finite_mask = np.isfinite(arr_display)
    if np.any(finite_mask):
        valid_data = arr_display[finite_mask]
        vmin = float(np.min(valid_data))
        vmax = float(np.max(valid_data))
        # If all values are the same, add small epsilon for contrast
        if vmin == vmax:
            vmin -= 1e-6
            vmax += 1e-6
    else:
        vmin, vmax = 0.0, 1.0
    
    # Final safety check - ensure vmin and vmax are strictly finite
    if not (np.isfinite(vmin) and np.isfinite(vmax)):
        vmin, vmax = 0.0, 1.0
    
    # Ensure vmin <= vmax
    if vmin > vmax:
        vmin, vmax = vmax, vmin
    
    if idx < 5:
        im = ax.imshow(arr_display, cmap='RdBu', origin='lower', vmin=vmin, vmax=vmax)
    else:
        im = ax.imshow(arr_display, cmap='viridis', origin='lower', vmin=vmin, vmax=vmax)
    
    cbar = plt.colorbar(im, ax=ax, location='bottom', pad=0.06, shrink=0.85)
    cbar.set_label(label, fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    
    ax.set_title(label, fontweight='bold', fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Print stats
    finite_mask = np.isfinite(arr)
    if np.any(finite_mask):
        valid_data = arr[finite_mask]
        print(f"{label:15s}: min={np.min(valid_data):+.4e}, max={np.max(valid_data):+.4e}, " + 
              f"mean={np.mean(valid_data):+.4e}, std={np.std(valid_data):.4e}")
    else:
        print(f"{label:15s}: [No finite data]")

plt.tight_layout()
plt.savefig(out_dir / "00_raw_data_overview.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Raw data visualization saved to: {out_dir / '00_raw_data_overview.png'}")

In [ ]:
# Diagnostic check for NaN/Inf values
print("\n=== DATA QUALITY CHECK ===")
for name, arr in [('e_xx', e_xx), ('e_yy', e_yy), ('e_xy', e_xy), ('theta', theta), ('mask', mask), ('error', error)]:
    n_nan = np.isnan(arr).sum()
    n_inf = np.isinf(arr).sum()
    n_finite = np.isfinite(arr).sum()
    print(f"{name:10s}: NaN={n_nan:6d}, Inf={n_inf:6d}, Finite={n_finite:6d} / {arr.size:6d}")

print("="*40)

In [ ]:
# ============================================================================
# SECTION 3: State-of-the-Art PINN Architecture (SIREN + Skip Connections)
# ============================================================================

class SirenLayer(nn.Module):
    """SIREN layer with sine activation and proper weight initialization."""
    def __init__(self, in_features, out_features, is_first=False, omega_0=1.0):
        super().__init__()
        self.in_features = in_features
        self.is_first = is_first
        self.omega_0 = omega_0
        
        self.linear = nn.Linear(in_features, out_features)
        
        # Proper initialization (Sitzmann et al., 2020)
        with torch.no_grad():
            if is_first:
                self.linear.weight.uniform_(-1 / in_features, 1 / in_features)
            else:
                bound = np.sqrt(6 / in_features) / omega_0
                self.linear.weight.uniform_(-bound, bound)
    
    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))


class StrainPINN_SIREN(nn.Module):
    """State-of-the-art SIREN architecture with skip connections for strain maps."""
    def __init__(self, hidden_dim=128, num_layers=6, omega_0=1.0, skip_connections=True):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.skip_connections = skip_connections
        
        # First layer: (x,y) -> hidden
        self.first = SirenLayer(2, hidden_dim, is_first=True, omega_0=omega_0)
        
        # Hidden layers with optional skip connections
        self.hidden_layers = nn.ModuleList([
            SirenLayer(hidden_dim, hidden_dim, is_first=False, omega_0=omega_0)
            for _ in range(num_layers - 2)
        ])
        
        # Skip connection projection (if used)
        if self.skip_connections:
            self.skip_proj = nn.Linear(hidden_dim, hidden_dim)
        
        # Final output layer
        self.final = nn.Linear(hidden_dim, 4)  # e_xx, e_yy, e_xy, theta
        
        # Initialize final layer to near-zero
        with torch.no_grad():
            self.final.weight.uniform_(-np.sqrt(6 / hidden_dim), np.sqrt(6 / hidden_dim))
    
    def forward(self, x):
        """x shape: (N, 2)"""
        # First layer
        x_h = self.first(x)  # (N, hidden_dim)
        
        # Hidden layers with skip connections
        for i, layer in enumerate(self.hidden_layers):
            if self.skip_connections and i % 2 == 0 and i > 0:
                # Skip connection every 2 layers
                x_h_new = layer(x_h) + self.skip_proj(x_h)
            else:
                x_h_new = layer(x_h)
            x_h = x_h_new
        
        # Final output
        out = self.final(x_h)  # (N, 4)
        return out


class StrainPINN_Fourier(nn.Module):
    """Fourier feature network: encodes spatial input with Fourier features."""
    def __init__(self, hidden_dim=128, num_layers=6, B_scale=1.0):
        super().__init__()
        self.B_scale = B_scale
        # Fixed Fourier feature matrix
        self.register_buffer('B', torch.randn(2, hidden_dim // 2) * B_scale)
        
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            *[nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU()) 
              for _ in range(num_layers - 2)],
            nn.Linear(hidden_dim, 4),
        )
    
    def forward(self, x):
        """x shape: (N, 2)"""
        # Compute Fourier features: [sin(2π*B*x), cos(2π*B*x)]
        proj = torch.matmul(x, self.B)  # (N, hidden_dim//2)
        x_enc = torch.cat([torch.sin(2 * np.pi * proj), torch.cos(2 * np.pi * proj)], dim=1)  # (N, hidden_dim)
        return self.net(x_enc)


class StrainPINN_Standard(nn.Module):
    """Standard MLP baseline."""
    def __init__(self, hidden_dim=128, num_layers=6):
        super().__init__()
        layers = [nn.Linear(2, hidden_dim), nn.ReLU()]
        for _ in range(num_layers - 2):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(hidden_dim, 4))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)


def create_pinn_model(arch_type, hidden_dim, num_layers, skip_connections=False, omega_0=30.0):
    """Factory function to create PINN models."""
    if arch_type == "siren":
        return StrainPINN_SIREN(hidden_dim, num_layers, omega_0=omega_0, skip_connections=skip_connections)  # [FIX] ω₀ from config (was hard-coded 1.0)
    elif arch_type == "fourier":
        return StrainPINN_Fourier(hidden_dim, num_layers, B_scale=1.0)
    elif arch_type == "standard":
        return StrainPINN_Standard(hidden_dim, num_layers)
    else:
        raise ValueError(f"Unknown architecture: {arch_type}")


# Instantiate model and show summary
model_test = create_pinn_model(config["architecture"], config["hidden_dim"], 
                               config["num_layers"], config["skip_connections"],
                               omega_0=config["omega_0"])
print(f"\n✓ Model Architecture: {config['architecture'].upper()}")
print(f"  Parameters: {sum(p.numel() for p in model_test.parameters()):,}")
print(model_test)

In [ ]:
# ============================================================================
# SECTION 4: Sampling Strategies & Sparse Masks
# ============================================================================

def create_sparse_mask(mask_full, sampling_fraction, seed=SEED):
    """
    Create sparse training mask by randomly selecting sampling_fraction of valid points.
    
    Args:
        mask_full: Full binary mask (H, W)
        sampling_fraction: Fraction of valid points to use [0, 1]
        seed: Random seed for reproducibility
        
    Returns:
        sparse_mask: Binary mask (H, W) with same shape as input
        selected_indices: 1D array of flat indices where mask_sparse == 1
    """
    rng = np.random.RandomState(seed)
    
    # Get indices of all valid points
    valid_indices = np.where(mask_full.flatten())[0]
    n_valid = len(valid_indices)
    
    # Sample points
    n_samples = max(1, int(np.ceil(sampling_fraction * n_valid)))
    selected_flat_indices = rng.choice(valid_indices, n_samples, replace=False)
    
    # Create sparse mask
    sparse_mask_flat = np.zeros(mask_full.size, dtype=np.float32)
    sparse_mask_flat[selected_flat_indices] = 1.0
    sparse_mask = sparse_mask_flat.reshape(mask_full.shape)
    
    return sparse_mask, selected_flat_indices


# Create sparse masks for all sampling fractions
sparse_masks = {}
sampled_indices = {}

print("\n" + "="*70)
print("SAMPLING STRATEGY: SPARSE TRAINING MASKS")
print("="*70)
for frac in config["sampling_fractions"]:
    sparse_mask, indices = create_sparse_mask(mask, frac, seed=SEED + int(frac * 1000))
    sparse_masks[frac] = torch.tensor(sparse_mask.flatten(), dtype=torch.float32, device=device)
    sampled_indices[frac] = indices
    
    n_train = sparse_mask.sum()
    n_valid = mask.sum()
    print(f"  {frac*100:5.0f}%: {int(n_train):6,d} / {int(n_valid):6,d} points → spatial coverage: {100*n_train/mask.size:.1f}%")

# Visualize sampling patterns
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Sampling Strategies: Training Point Distribution', fontsize=14, fontweight='bold')

for idx, frac in enumerate(config["sampling_fractions"]):
    ax = axes.flat[idx]
    
    # Show the full mask in light gray
    ax.imshow(mask, cmap='gray', alpha=0.3, origin='lower')
    
    # Overlay sparse mask
    sparse_mask_np = sparse_masks[frac].cpu().numpy().reshape(H, W)
    ax.imshow(sparse_mask_np, cmap='Reds', alpha=0.7, origin='lower')
    
    n_train = sparse_mask_np.sum()
    ax.set_title(f'{frac*100:.0f}% Sampling\n({int(n_train):,} points)', fontweight='bold', fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.savefig(out_dir / "01_sampling_patterns.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Sampling visualization saved to: {out_dir / '01_sampling_patterns.png'}")

In [ ]:
# ============================================================================
# SECTION 5: Physics-Informed Loss (autodiff @ collocation points)
#   - Elastic equilibrium  div(sigma) = 0   (Hooke's law, plane stress)
#   - Saint-Venant compatibility  d2_yy e_xx + d2_xx e_yy - 2 d2_xy e_xy = 0
# Spatial derivatives are obtained by reverse-mode automatic differentiation of
# the network w.r.t. its input coordinate (no finite-difference stencil).
# ============================================================================

def compute_stress_from_strain(e_xx, e_yy, e_xy, E, nu):
    """Plane-stress linear elasticity.
        sigma_xx = E/(1-nu^2) (e_xx + nu e_yy)
        sigma_yy = E/(1-nu^2) (e_yy + nu e_xx)
        sigma_xy = E/(2(1+nu)) e_xy = G e_xy
    """
    factor = E / (1.0 - nu ** 2)
    sigma_xx = factor * (e_xx + nu * e_yy)
    sigma_yy = factor * (e_yy + nu * e_xx)
    G = E / (2.0 * (1.0 + nu))
    sigma_xy = G * e_xy
    return sigma_xx, sigma_yy, sigma_xy


def _grad(outputs, inputs):
    """First derivative d outputs / d inputs with graph retained for higher order."""
    return torch.autograd.grad(
        outputs, inputs, grad_outputs=torch.ones_like(outputs),
        create_graph=True, retain_graph=True)[0]


def physics_residuals_collocation(model, n_coll, device, cfg, extra_pts=None):
    """Equilibrium + compatibility residuals evaluated by autodiff at random
    collocation points in the unit square (normalised coordinates).

    Residuals are formed on the network's (scaled) strain outputs with the
    elastic operator non-dimensionalised by E/(1-nu^2) ('normalize_physics_by_E'),
    so the equilibrium and compatibility terms are O(1) and commensurate with the
    data MSE. Returns (L_equil, L_compat) as scalar mean-squared residuals.
    """
    nu = cfg["nu"]
    pts = torch.rand(n_coll, 2, device=device)
    if extra_pts is not None and extra_pts.numel() > 0:
        pts = torch.cat([pts, extra_pts.to(device)], dim=0)
    pts = pts.detach().requires_grad_(True)

    out = model(pts)
    e_xx, e_yy, e_xy = out[:, 0], out[:, 1], out[:, 2]

    g_xx = _grad(e_xx, pts); g_yy = _grad(e_yy, pts); g_xy = _grad(e_xy, pts)
    exx_x, exx_y = g_xx[:, 0], g_xx[:, 1]
    eyy_x, eyy_y = g_yy[:, 0], g_yy[:, 1]
    exy_x, exy_y = g_xy[:, 0], g_xy[:, 1]

    # dimensionless stress gradients (sigma / [E/(1-nu^2)])
    sxx_x = exx_x + nu * eyy_x
    syy_y = eyy_y + nu * exx_y
    syy_x = eyy_x + nu * exx_x
    c_xy  = (1.0 - nu) * 0.5                # sigma_xy / [E/(1-nu^2)] = (1-nu)/2 * e_xy
    sxy_x = c_xy * exy_x
    sxy_y = c_xy * exy_y

    eq1 = sxx_x + sxy_y                      # d_x sigma_xx + d_y sigma_xy
    eq2 = sxy_x + syy_y                      # d_x sigma_xy + d_y sigma_yy
    L_equil = (eq1 ** 2 + eq2 ** 2).mean()

    # second derivatives for Saint-Venant compatibility
    exx_yy = _grad(exx_y, pts)[:, 1]
    eyy_xx = _grad(eyy_x, pts)[:, 0]
    exy_xy = _grad(exy_x, pts)[:, 1]
    L_compat = (exx_yy + eyy_xx - 2.0 * exy_xy).pow(2).mean()

    return L_equil, L_compat


def grid_physics_diagnostics(e_xx_map, e_yy_map, e_xy_map, valid_mask, cfg):
    """Physical-unit physics-consistency diagnostics on a dense (H,W) strain map.
    Uses central finite differences on the reconstructed grid so the SAME operator
    can score networks (PINN, SIREN-only) and non-network baselines (CS, GP).

        R_equil  : sqrt(mean[ (d_x sx + d_y sxy)^2 + (d_x sxy + d_y syy)^2 ])  [GPa/um]
        R_compat : sqrt(mean[ (d2_yy exx + d2_xx eyy - 2 d2_xy exy)^2 ])       [1/um^2]
    """
    E_GPa = cfg["E"] / 1e9
    nu    = cfg["nu"]
    px    = cfg["px_um"]                      # um/pixel  -> derivatives per um

    sxx, syy, sxy = compute_stress_from_strain(e_xx_map, e_yy_map, e_xy_map, E_GPa, nu)
    # np.gradient(.., px) returns (d/dy, d/dx) for a (row=y, col=x) array
    dsxx_dy, dsxx_dx = np.gradient(sxx, px)
    dsyy_dy, dsyy_dx = np.gradient(syy, px)
    dsxy_dy, dsxy_dx = np.gradient(sxy, px)
    eq1 = dsxx_dx + dsxy_dy
    eq2 = dsxy_dx + dsyy_dy

    exx_y, exx_x = np.gradient(e_xx_map, px)
    eyy_y, eyy_x = np.gradient(e_yy_map, px)
    exy_y, exy_x = np.gradient(e_xy_map, px)
    exx_yy = np.gradient(exx_y, px, axis=0)
    eyy_xx = np.gradient(eyy_x, px, axis=1)
    exy_xy = np.gradient(exy_x, px, axis=0)
    comp = exx_yy + eyy_xx - 2.0 * exy_xy

    m = valid_mask > 0.5
    R_equil  = float(np.sqrt(np.mean(eq1[m] ** 2 + eq2[m] ** 2)))
    R_compat = float(np.sqrt(np.mean(comp[m] ** 2)))
    return R_equil, R_compat


def normalized_physics(L_eq, L_co, cfg, ema_state):
    """[FIX] Divide each residual by a running (EMA) estimate of its own magnitude,
    with the EMA FROZEN after cfg["phys_norm_warmup"] steps. The warmup sets the
    scale so physics gradients are commensurate with the O(1) data-MSE term
    (raw residuals on standardised fields are O(1e2)-O(1e4) and otherwise crush
    the data term); freezing afterwards makes the physics term a fixed-scale
    penalty whose pull decays as the residual shrinks. A live EMA keeps the pull
    constant — equivalent to minimising log L — and eventually grinds the network
    into the trivial flat-field solution once the data loss plateaus.
    Set cfg["normalize_residuals"]=False to reproduce the original behaviour."""
    if not cfg.get("normalize_residuals", False):
        return cfg["alpha_eq"] * L_eq + cfg["alpha_co"] * L_co
    ema_state["n"] = ema_state.get("n", 0) + 1
    if ema_state["n"] <= cfg.get("phys_norm_warmup", 200):
        for key, L in (("eq", L_eq), ("co", L_co)):
            v = float(L.detach())
            ema_state[key] = v if key not in ema_state else 0.99 * ema_state[key] + 0.01 * v
    return (cfg["alpha_eq"] * L_eq / (ema_state["eq"] + 1e-12)
            + cfg["alpha_co"] * L_co / (ema_state["co"] + 1e-12))


def pinn_loss_with_physics(model, x, y, e_xx_true, e_yy_true, e_xy_true, theta_true,
                           sparse_mask, H, W, device, cfg, ema_state=None, lam_ramp=1.0):
    """Fixed-weight data + collocation-physics loss.
    Retained for the Bayesian variants (Section 10); the deterministic training
    loop below uses its own ramped objective.
    """
    idx = sparse_mask > 0.5
    inp = torch.stack([x[idx], y[idx]], dim=1)
    out = model(inp)
    tgt = torch.stack([e_xx_true[idx], e_yy_true[idx], e_xy_true[idx], theta_true[idx]], dim=1)
    loss_data = ((out - tgt) ** 2).mean()

    L_eq, L_co = physics_residuals_collocation(model, cfg["n_collocation"], device, cfg)
    loss_phys = normalized_physics(L_eq, L_co, cfg, ema_state if ema_state is not None else {})  # [FIX]
    total = cfg["w_data"] * loss_data + lam_ramp * cfg["lambda_phys_max"] * loss_phys  # [FIX] ramped
    return total, loss_data.detach(), loss_phys.detach()


print("✓ Physics-informed loss (autodiff equilibrium + Saint-Venant compatibility) defined")


In [ ]:
# ============================================================================
# SECTION 6: Residual-Based Adaptive Refinement (RAR) of collocation points
# Concentrates collocation density where the PDE residual is largest, the PINN
# analogue of adaptive mesh refinement. Operates on PHYSICS collocation points
# (it does not add measured data to the training set).
# ============================================================================

def adaptive_collocation_refinement(model, cfg, device, n_candidates=5000):
    """Draw a large random candidate set, score each point by its elastic-
    equilibrium residual, and return the top-(rar_pct * n_collocation) points to
    append to the persistent collocation pool.
    """
    nu = cfg["nu"]
    pts = torch.rand(n_candidates, 2, device=device, requires_grad=True)
    out = model(pts)
    e_xx, e_yy, e_xy = out[:, 0], out[:, 1], out[:, 2]
    g_xx = _grad(e_xx, pts); g_yy = _grad(e_yy, pts); g_xy = _grad(e_xy, pts)
    sxx_x = g_xx[:, 0] + nu * g_yy[:, 0]
    syy_y = g_yy[:, 1] + nu * g_xx[:, 1]
    c_xy  = (1.0 - nu) * 0.5
    sxy_x = c_xy * g_xy[:, 0]
    sxy_y = c_xy * g_xy[:, 1]
    resid = (sxx_x + sxy_y) ** 2 + (sxy_x + syy_y) ** 2

    k = max(1, int(cfg["rar_num_new_points_pct"] * cfg["n_collocation"]))
    k = min(k, resid.numel())
    idx = torch.topk(resid.detach(), k).indices
    return pts.detach()[idx]


print("✓ Residual-based adaptive collocation refinement (RAR) defined")


In [ ]:
# ============================================================================
# SECTION 7: Main Training Loop (ramped physics weight + RAR + early stopping)
# ============================================================================

def train_pinn_sparse(sampling_fraction, sparse_mask, cfg, device, lambda_phys_scale=1.0):
    """Train one SIREN PINN on sparse data.

    lambda_phys_scale : multiplies the physics weight. Use 1.0 for the full PINN
                        and 0.0 for the data-only SIREN ablation (Section 4).

    Composite loss (Eq.):
        L = w_data * L_data
            + lambda_phys(t) * (alpha_eq * L_equil + alpha_co * L_compat)
        lambda_phys(t) = lambda_phys_max * (1 - exp(-t / tau)) * lambda_phys_scale

    Early stopping is on a held-out VALIDATION subset of the sampled points.
    """
    torch.manual_seed(SEED)
    model = create_pinn_model(cfg["architecture"], cfg["hidden_dim"],
                              cfg["num_layers"], cfg["skip_connections"],
                              omega_0=cfg["omega_0"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=1e-6)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=cfg["lr_decay_interval"],
                                            gamma=cfg["lr_decay_factor"])

    # ---- train / validation split of the sampled probe positions ----
    samp_idx = torch.where(sparse_mask > 0.5)[0].cpu().numpy()
    rng = np.random.RandomState(SEED)
    rng.shuffle(samp_idx)
    n_val = max(1, int(cfg["val_fraction"] * len(samp_idx)))
    val_idx = samp_idx[:n_val]
    tr_idx  = samp_idx[n_val:] if len(samp_idx) - n_val > 0 else samp_idx
    tr_idx  = torch.as_tensor(tr_idx, dtype=torch.long, device=device)
    val_idx = torch.as_tensor(val_idx, dtype=torch.long, device=device)

    coords_all = torch.stack([x_flat, y_flat], dim=1)
    tr_in  = coords_all[tr_idx]
    val_in = coords_all[val_idx]
    tgt_all = torch.stack([e_xx_scaled, e_yy_scaled, e_xy_scaled, theta_scaled], dim=1)
    tr_tgt  = tgt_all[tr_idx]
    val_tgt = tgt_all[val_idx]

    history = {"epoch": [], "loss_total": [], "loss_data": [],
               "loss_physics": [], "val_loss": [], "lr": [], "lambda_phys": []}

    best_val = float("inf"); best_state = None; patience_counter = 0
    coll_pool = torch.empty(0, 2, device=device)
    ema_state = {}                      # [FIX] running scale of the physics residuals
    use_phys = lambda_phys_scale > 0.0

    desc = f"Train {sampling_fraction*100:.0f}%" + ("" if use_phys else " [data-only]")
    for epoch in trange(cfg["epochs"], desc=desc, leave=False):
        # RAR: grow the persistent collocation pool at high-residual regions
        if (use_phys and cfg["enable_rar"] and epoch >= cfg["rar_trigger_epoch"]
                and epoch % cfg["rar_interval"] == 0):
            new_pts = adaptive_collocation_refinement(model, cfg, device)
            coll_pool = torch.cat([coll_pool, new_pts], dim=0)
            cap = cfg["rar_max_pool_mult"] * cfg["n_collocation"]
            if coll_pool.shape[0] > cap:
                coll_pool = coll_pool[-cap:]

        model.train(); opt.zero_grad()
        pred = model(tr_in)
        loss_data = ((pred - tr_tgt) ** 2).mean()

        if use_phys:
            lam = cfg["lambda_phys_max"] * (1.0 - np.exp(-epoch / cfg["ramp_tau"])) * lambda_phys_scale
            L_eq, L_co = physics_residuals_collocation(
                model, cfg["n_collocation"], device, cfg,
                extra_pts=(coll_pool if coll_pool.numel() else None))
            loss_phys = normalized_physics(L_eq, L_co, cfg, ema_state)  # [FIX]
            total = cfg["w_data"] * loss_data + lam * loss_phys
        else:
            lam = 0.0
            loss_phys = torch.zeros((), device=device)
            total = cfg["w_data"] * loss_data

        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step(); sched.step()

        with torch.no_grad():
            model.eval()
            val_loss = ((model(val_in) - val_tgt) ** 2).mean().item()

        history["epoch"].append(epoch)
        history["loss_total"].append(total.item())
        history["loss_data"].append(loss_data.item())
        history["loss_physics"].append(float(loss_phys.item()))
        history["val_loss"].append(val_loss)
        history["lr"].append(sched.get_last_lr()[0])
        history["lambda_phys"].append(float(lam))

        if (best_val - val_loss) > cfg["min_delta"]:
            best_val = val_loss; patience_counter = 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
        if cfg["early_stop"] and patience_counter >= cfg["patience"]:
            if cfg["verbose"]:
                print(f"      -> early stop @ epoch {epoch} (best val loss {best_val:.4e})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("TRAINING: PINNs across sampling fractions (autodiff physics + ramp + RAR)")
print("="*70)

trained_models     = {}
training_histories = {}
training_times     = {}

start_time_total = time.time()
for frac in config["sampling_fractions"]:
    print(f"\n▶ TRAINING on {frac*100:.0f}% data ...")
    t0 = time.time()
    model, history = train_pinn_sparse(frac, sparse_masks[frac], config, device,
                                       lambda_phys_scale=1.0)
    elapsed = time.time() - t0
    trained_models[frac]     = model
    training_histories[frac] = history
    training_times[frac]     = elapsed
    print(f"  ✓ {elapsed:.1f}s | {len(history['epoch'])} epochs | "
          f"final loss {history['loss_total'][-1]:.4e} | best val {min(history['val_loss']):.4e}")

total_time = time.time() - start_time_total
print(f"\n✓ All PINN training completed in {total_time:.1f}s")


In [ ]:
# ============================================================================
# SECTION 8: Evaluation & Metrics Computation
# ============================================================================

def evaluate_model(model, x, y, e_xx_true, e_yy_true, e_xy_true, theta_true,
                   mask_eval, H, W, device):
    """
    Evaluate model and compute comprehensive metrics.
    
    Returns:
        predictions: Dict of predicted strain components
        metrics: Dict of per-component metrics
    """
    with torch.no_grad():
        inputs = torch.stack([x, y], dim=1)
        outputs = model(inputs)
        e_xx_pred_s, e_yy_pred_s, e_xy_pred_s, theta_pred_s = outputs[:, 0], outputs[:, 1], outputs[:, 2], outputs[:, 3]
    
    # Unscale predictions
    e_xx_pred = unscale(e_xx_pred_s, 'e_xx')
    e_yy_pred = unscale(e_yy_pred_s, 'e_yy')
    e_xy_pred = unscale(e_xy_pred_s, 'e_xy')
    theta_pred = unscale(theta_pred_s, 'theta')
    
    # Move to CPU for numpy operations
    e_xx_pred_np = e_xx_pred.cpu().numpy().reshape(H, W)
    e_yy_pred_np = e_yy_pred.cpu().numpy().reshape(H, W)
    e_xy_pred_np = e_xy_pred.cpu().numpy().reshape(H, W)
    theta_pred_np = theta_pred.cpu().numpy().reshape(H, W)
    
    predictions = {
        'e_xx': e_xx_pred_np,
        'e_yy': e_yy_pred_np,
        'e_xy': e_xy_pred_np,
        'theta': theta_pred_np,
    }
    
    # Compute metrics per component
    metrics = {}
    for name, pred, true in [
        ('e_xx', e_xx_pred_np, e_xx),
        ('e_yy', e_yy_pred_np, e_yy),
        ('e_xy', e_xy_pred_np, e_xy),
        ('theta', theta_pred_np, theta),
    ]:
        valid_idx = mask_eval > 0.5
        pred_valid = pred[valid_idx]
        true_valid = true[valid_idx]
        
        mse = np.mean((pred_valid - true_valid) ** 2)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(pred_valid - true_valid))
        max_err = np.max(np.abs(pred_valid - true_valid))
        
        # R² score
        ss_res = np.sum((true_valid - pred_valid) ** 2)
        ss_tot = np.sum((true_valid - np.mean(true_valid)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
        
        # Normalized RMSE
        range_true = np.max(true_valid) - np.min(true_valid)
        nrmse_range = rmse / range_true if range_true > 0 else 0.0
        
        std_true = np.std(true_valid)
        nrmse_std = rmse / std_true if std_true > 0 else 0.0
        
        metrics[name] = {
            'MSE': mse,
            'RMSE': rmse,
            'MAE': mae,
            'Max Error': max_err,
            'R²': r2,
            'NRMSE (range)': nrmse_range,
            'NRMSE (std)': nrmse_std,
        }
    
    return predictions, metrics


# Evaluate all models
print("\n" + "="*70)
print("EVALUATION: Computing Metrics Across All Models")
print("="*70)

all_predictions = {}
all_metrics = {}

for frac in config["sampling_fractions"]:
    print(f"  Evaluating {frac*100:.0f}%...")
    preds, metrics = evaluate_model(trained_models[frac], x_flat, y_flat,
                                   e_xx, e_yy, e_xy, theta,
                                   mask, H, W, device)
    all_predictions[frac] = preds
    all_metrics[frac] = metrics

print("✓ Evaluation complete")

# Create metrics summary table
print("\nMetrics Summary (R² Score):")
r2_summary = pd.DataFrame({
    'Sampling %': [f'{f*100:.0f}%' for f in config["sampling_fractions"]],
    'e_xx': [all_metrics[f]['e_xx']['R²'] for f in config["sampling_fractions"]],
    'e_yy': [all_metrics[f]['e_yy']['R²'] for f in config["sampling_fractions"]],
    'e_xy': [all_metrics[f]['e_xy']['R²'] for f in config["sampling_fractions"]],
    'theta': [all_metrics[f]['theta']['R²'] for f in config["sampling_fractions"]],
})
print(r2_summary.to_string(index=False))

print("\nMetrics Summary (RMSE):")
rmse_summary = pd.DataFrame({
    'Sampling %': [f'{f*100:.0f}%' for f in config["sampling_fractions"]],
    'e_xx': [all_metrics[f]['e_xx']['RMSE'] for f in config["sampling_fractions"]],
    'e_yy': [all_metrics[f]['e_yy']['RMSE'] for f in config["sampling_fractions"]],
    'e_xy': [all_metrics[f]['e_xy']['RMSE'] for f in config["sampling_fractions"]],
    'theta': [all_metrics[f]['theta']['RMSE'] for f in config["sampling_fractions"]],
})
print(rmse_summary.to_string(index=False))

In [ ]:
# ============================================================================
# SECTION 9: Visualization & Comparison
# ============================================================================

# 1. Loss curves for each sampling fraction
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Dynamics: Loss Convergence', fontsize=14, fontweight='bold')

ax = axes[0]
for frac in config["sampling_fractions"]:
    history = training_histories[frac]
    ax.semilogy(history['epoch'], history['loss_total'], marker='o', markersize=2, 
               label=f'{frac*100:.0f}%', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Total Loss (log scale)', fontsize=12)
ax.set_title('Convergence vs. Sampling Fraction', fontweight='bold')
ax.legend(loc='best', ncol=2)
ax.grid(True, alpha=0.3)

# Data vs. Physics loss breakdown for one model (50%)
ax = axes[1]
frac = 0.50
history = training_histories[frac]
ax.semilogy(history['epoch'], history['loss_data'], label='Data Loss', linewidth=2.5, color='green')
ax.semilogy(history['epoch'], history['loss_physics'], label='Physics Loss', linewidth=2.5, color='red')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (log scale)', fontsize=12)
ax.set_title(f'Loss Components @ 50% Sampling', fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(out_dir / "02_loss_convergence.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Loss curves saved to: {out_dir / '02_loss_convergence.png'}")

In [ ]:
# 2. Side-by-side predictions for selected fractions with consistent colorbars
selected_fracs = [0.01, 0.10, 0.50]

# Pixel-to-nm calibration from metadata
# Real space pixel size: 0.017872741 µm = 17.872741 nm/pixel
calibration_nm_per_pixel = 0.017872741 * 1000  # Convert µm to nm

for component, true_data in [('e_xx', e_xx), ('e_yy', e_yy), ('e_xy', e_xy)]:
    fig, axes = plt.subplots(1, len(selected_fracs) + 1, figsize=(18, 5))
    fig.suptitle(f'Predictions: {component}', fontsize=14, fontweight='bold')
    
    # Compute consistent vmin/vmax across ALL data (ground truth + all predictions)
    all_data = [true_data] + [all_predictions[frac][component] for frac in selected_fracs]
    valid_mask = np.isfinite(true_data)
    vmin = np.nanmin([np.nanmin(arr[valid_mask]) for arr in all_data])
    vmax = np.nanmax([np.nanmax(arr[valid_mask]) for arr in all_data])
    
    # Add 5% padding to the range for better contrast
    range_val = vmax - vmin
    vmin -= 0.05 * range_val
    vmax += 0.05 * range_val
    
    # Ground truth
    ax = axes[0]
    im = ax.imshow(true_data, cmap='RdBu', origin='lower', vmin=vmin, vmax=vmax)
    cbar = plt.colorbar(im, ax=ax, location='bottom', pad=0.1, fraction=0.046)
    cbar.set_label(component, fontsize=10)
    ax.set_title('Ground Truth', fontweight='bold', fontsize=12)
    # ax.set_xlabel('x (nm)', fontsize=9)
    # ax.set_ylabel('y (nm)', fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Add scale bar to ground truth (in nm, no label)
    scale_length_px = W // 8  # 1/8 of width in pixels
    scale_length_nm = scale_length_px * calibration_nm_per_pixel
    bar_x_start = W - scale_length_px - 20
    bar_x_end = W - 20
    bar_y = H - 20
    ax.plot([bar_x_start, bar_x_end], [bar_y, bar_y], 'k-', linewidth=8)
    
    # Predictions at each sampling fraction
    for idx, frac in enumerate(selected_fracs):
        ax = axes[idx + 1]
        pred = all_predictions[frac][component]
        im = ax.imshow(pred, cmap='RdBu', origin='lower', vmin=vmin, vmax=vmax)
        cbar = plt.colorbar(im, ax=ax, location='bottom', pad=0.1, fraction=0.046)
        cbar.set_label(component, fontsize=10)
        
        r2 = all_metrics[frac][component]['R²']
        rmse = all_metrics[frac][component]['RMSE']
        ax.set_title(f'{frac*100:.0f}% Data\nR²={r2:.3f}, RMSE={rmse:.2e}', 
                     fontweight='bold', fontsize=12)
        # ax.set_xlabel('x (nm)', fontsize=9)
        # ax.set_ylabel('y (nm)', fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        
        # Add scale bar (in nm, no label)
        ax.plot([bar_x_start, bar_x_end], [bar_y, bar_y], 'k-', linewidth=8)
    
    plt.tight_layout()
    plt.savefig(out_dir / f"03_predictions_{component}.png", dpi=300, bbox_inches='tight')
    plt.show()

print(f"✓ Prediction maps with consistent colorbars and scale bars (nm) saved")
print(f"  Calibration: {calibration_nm_per_pixel:.2f} nm/pixel")

In [ ]:
# 3. Error heatmaps
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Absolute Error Maps: e_xx', fontsize=14, fontweight='bold')

for idx, frac in enumerate(config["sampling_fractions"]):
    ax = axes.flat[idx]
    error_map = np.abs(all_predictions[frac]['e_xx'] - e_xx)
    im = ax.imshow(error_map, cmap='hot', origin='lower')
    cbar = plt.colorbar(im, ax=ax, location='bottom', pad=0.06, shrink=0.85)
    cbar.set_label('|Error|', fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    
    mae = all_metrics[frac]['e_xx']['MAE']
    max_err = all_metrics[frac]['e_xx']['Max Error']
    ax.set_title(f'{frac*100:.0f}%  MAE={mae:.2e}', fontweight='bold', fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.savefig(out_dir / "04_error_heatmaps_exx.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Error heatmaps saved")

In [ ]:
# ============================================================================
# Overlay Sampling Points on Predictions
# ============================================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Predictions with Training Point Overlays (e_xx)', fontsize=14, fontweight='bold')

selected_fracs = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75]

# Pixel-to-nm calibration
calibration_nm_per_pixel = 0.017872741 * 1000  # Convert µm to nm

# Compute consistent colorbar range across all fractions
all_preds = [all_predictions[frac]['e_xx'] for frac in selected_fracs]
valid_mask = mask > 0.5
all_valid_preds = np.concatenate([pred[valid_mask] for pred in all_preds])
vmin_global = np.nanmin(all_valid_preds)
vmax_global = np.nanmax(all_valid_preds)

for idx, frac in enumerate(selected_fracs):
    ax = axes.flat[idx]
    
    # Get predictions for this fraction
    e_xx_pred = all_predictions[frac]['e_xx']
    
    # Show prediction map with consistent colorbar range
    im = ax.imshow(e_xx_pred, cmap='RdBu', origin='lower', vmin=vmin_global, vmax=vmax_global, alpha=0.9)
    
    # Overlay training points as black dots
    sparse_mask_np = sparse_masks[frac].cpu().numpy().reshape(H, W)
    y_pts, x_pts = np.where(sparse_mask_np > 0.5)
    ax.scatter(x_pts, y_pts, c='black', s=5, alpha=0.5, edgecolors='white', linewidths=0.3, label='Training points')

    # Add colorbar with larger label fontsize
    cbar = plt.colorbar(im, ax=ax, label='e_xx', location='bottom', pad=0.1)
    cbar.set_label('e_xx', fontsize=12, fontweight='bold')
    
    n_train = int(sparse_mask_np.sum())
    ax.set_title(f'{frac*100:.0f}% Sampling ({n_train:,} points)\nR²={all_metrics[frac]["e_xx"]["R²"]:.3f}', 
                 fontweight='bold', fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc='upper left', fontsize=14)
    
    # Add scale bar (in nm, no label)
    scale_length_px = W // 8  # 1/8 of width in pixels
    bar_x_start = W - scale_length_px - 20
    bar_x_end = W - 20
    bar_y = H - 20
    ax.plot([bar_x_start, bar_x_end], [bar_y, bar_y], 'k-', linewidth=8)

plt.tight_layout()
plt.savefig(out_dir / "07_predictions_with_sampling_overlay.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Predictions with sampling overlay saved to: {out_dir / '07_predictions_with_sampling_overlay.png'}")
print(f"  Calibration: {calibration_nm_per_pixel:.2f} nm/pixel")
print(f"  Colorbar range: {vmin_global:.4e} to {vmax_global:.4e} (consistent across all fractions)")


In [ ]:
# ============================================================================
# Overlay Sampling Points on Predictions (e_yy)
# ============================================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Predictions with Training Point Overlays (e_yy)', fontsize=14, fontweight='bold')

selected_fracs = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75]

# Pixel-to-nm calibration
calibration_nm_per_pixel = 0.017872741 * 1000  # Convert µm to nm

# Compute consistent colorbar range across all fractions
all_preds_eyy = [all_predictions[frac]['e_yy'] for frac in selected_fracs]
valid_mask = mask > 0.5
all_valid_preds_eyy = np.concatenate([pred[valid_mask] for pred in all_preds_eyy])
vmin_global_eyy = np.nanmin(all_valid_preds_eyy)
vmax_global_eyy = np.nanmax(all_valid_preds_eyy)

for idx, frac in enumerate(selected_fracs):
    ax = axes.flat[idx]
    
    # Get predictions for this fraction
    e_yy_pred = all_predictions[frac]['e_yy']
    
    # Show prediction map with consistent colorbar range
    im = ax.imshow(e_yy_pred, cmap='RdBu', origin='lower', vmin=vmin_global_eyy, vmax=vmax_global_eyy, alpha=0.9)
    
    # Overlay training points as black dots
    sparse_mask_np = sparse_masks[frac].cpu().numpy().reshape(H, W)
    y_pts, x_pts = np.where(sparse_mask_np > 0.5)
    ax.scatter(x_pts, y_pts, c='black', s=5, alpha=0.5, edgecolors='white', linewidths=0.3, label='Training points')

    # Add colorbar with larger label fontsize
    cbar = plt.colorbar(im, ax=ax, label='e_yy', location='bottom', pad=0.1)
    cbar.set_label('e_yy', fontsize=12, fontweight='bold')
    
    n_train = int(sparse_mask_np.sum())
    ax.set_title(f'{frac*100:.0f}% Sampling ({n_train:,} points)\nR²={all_metrics[frac]["e_yy"]["R²"]:.3f}', 
                 fontweight='bold', fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc='upper left', fontsize=14)
    
    # Add scale bar (in nm, no label)
    scale_length_px = W // 8  # 1/8 of width in pixels
    bar_x_start = W - scale_length_px - 20
    bar_x_end = W - 20
    bar_y = H - 20
    ax.plot([bar_x_start, bar_x_end], [bar_y, bar_y], 'k-', linewidth=8)

plt.tight_layout()
plt.savefig(out_dir / "07b_predictions_with_sampling_overlay_eyy.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Predictions with sampling overlay saved to: {out_dir / '07b_predictions_with_sampling_overlay_eyy.png'}")
print(f"  Calibration: {calibration_nm_per_pixel:.2f} nm/pixel")
print(f"  Colorbar range: {vmin_global_eyy:.4e} to {vmax_global_eyy:.4e} (consistent across all fractions)")


# ============================================================================
# Overlay Sampling Points on Predictions (e_xy)
# ============================================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Predictions with Training Point Overlays (e_xy)', fontsize=14, fontweight='bold')

# Compute consistent colorbar range across all fractions
all_preds_exy = [all_predictions[frac]['e_xy'] for frac in selected_fracs]
all_valid_preds_exy = np.concatenate([pred[valid_mask] for pred in all_preds_exy])
vmin_global_exy = np.nanmin(all_valid_preds_exy)
vmax_global_exy = np.nanmax(all_valid_preds_exy)

for idx, frac in enumerate(selected_fracs):
    ax = axes.flat[idx]
    
    # Get predictions for this fraction
    e_xy_pred = all_predictions[frac]['e_xy']
    
    # Show prediction map with consistent colorbar range
    im = ax.imshow(e_xy_pred, cmap='RdBu', origin='lower', vmin=vmin_global_exy, vmax=vmax_global_exy, alpha=0.9)
    
    # Overlay training points as black dots
    sparse_mask_np = sparse_masks[frac].cpu().numpy().reshape(H, W)
    y_pts, x_pts = np.where(sparse_mask_np > 0.5)
    ax.scatter(x_pts, y_pts, c='black', s=5, alpha=0.5, edgecolors='white', linewidths=0.3, label='Training points')

    # Add colorbar with larger label fontsize
    cbar = plt.colorbar(im, ax=ax, label='e_xy', location='bottom', pad=0.1)
    cbar.set_label('e_xy', fontsize=12, fontweight='bold')
    
    n_train = int(sparse_mask_np.sum())
    ax.set_title(f'{frac*100:.0f}% Sampling ({n_train:,} points)\nR²={all_metrics[frac]["e_xy"]["R²"]:.3f}', 
                 fontweight='bold', fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc='upper left', fontsize=14)
    
    # Add scale bar (in nm, no label)
    scale_length_px = W // 8  # 1/8 of width in pixels
    bar_x_start = W - scale_length_px - 20
    bar_x_end = W - 20
    bar_y = H - 20
    ax.plot([bar_x_start, bar_x_end], [bar_y, bar_y], 'k-', linewidth=8)

plt.tight_layout()
plt.savefig(out_dir / "07c_predictions_with_sampling_overlay_exy.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Predictions with sampling overlay saved to: {out_dir / '07c_predictions_with_sampling_overlay_exy.png'}")
print(f"  Calibration: {calibration_nm_per_pixel:.2f} nm/pixel")
print(f"  Colorbar range: {vmin_global_exy:.4e} to {vmax_global_exy:.4e} (consistent across all fractions)")


# ============================================================================
# Overlay Sampling Points on Predictions (theta - rotation angle)
# ============================================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Predictions with Training Point Overlays (θ - Rotation Angle)', fontsize=14, fontweight='bold')

# Compute consistent colorbar range across all fractions
all_preds_theta = [all_predictions[frac]['theta'] for frac in selected_fracs]
all_valid_preds_theta = np.concatenate([pred[valid_mask] for pred in all_preds_theta])
vmin_global_theta = np.nanmin(all_valid_preds_theta)
vmax_global_theta = np.nanmax(all_valid_preds_theta)

for idx, frac in enumerate(selected_fracs):
    ax = axes.flat[idx]
    
    # Get predictions for this fraction
    theta_pred = all_predictions[frac]['theta']
    
    # Show prediction map with consistent colorbar range
    im = ax.imshow(theta_pred, cmap='hsv', origin='lower', vmin=vmin_global_theta, vmax=vmax_global_theta, alpha=0.9)
    
    # Overlay training points as black dots
    sparse_mask_np = sparse_masks[frac].cpu().numpy().reshape(H, W)
    y_pts, x_pts = np.where(sparse_mask_np > 0.5)
    ax.scatter(x_pts, y_pts, c='white', s=5, alpha=0.5, edgecolors='black', linewidths=0.3, label='Training points')

    # Add colorbar with larger label fontsize
    cbar = plt.colorbar(im, ax=ax, label='θ (rad)', location='bottom', pad=0.1)
    cbar.set_label('θ (rad)', fontsize=12, fontweight='bold')
    
    n_train = int(sparse_mask_np.sum())
    ax.set_title(f'{frac*100:.0f}% Sampling ({n_train:,} points)\nR²={all_metrics[frac]["theta"]["R²"]:.3f}', 
                 fontweight='bold', fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc='upper left', fontsize=14)
    
    # Add scale bar (in nm, no label)
    scale_length_px = W // 8  # 1/8 of width in pixels
    bar_x_start = W - scale_length_px - 20
    bar_x_end = W - 20
    bar_y = H - 20
    ax.plot([bar_x_start, bar_x_end], [bar_y, bar_y], 'k-', linewidth=8)

plt.tight_layout()
plt.savefig(out_dir / "07d_predictions_with_sampling_overlay_theta.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Predictions with sampling overlay saved to: {out_dir / '07d_predictions_with_sampling_overlay_theta.png'}")
print(f"  Calibration: {calibration_nm_per_pixel:.2f} nm/pixel")
print(f"  Colorbar range: {vmin_global_theta:.4e} to {vmax_global_theta:.4e} (consistent across all fractions)")

In [ ]:
# ============================================================================
# Combined Predictions: e_xx, e_yy, e_xy at 1%, 5%, 10% Sampling
# ============================================================================

fig, axes = plt.subplots(3, 3, figsize=(16, 14))
fig.suptitle('Strain Component Predictions at Low Sampling Fractions (1%, 5%, 10%)', 
             fontsize=16, fontweight='bold')

selected_fracs = [0.01, 0.05, 0.10]
components = ['e_xx', 'e_yy', 'e_xy']

# Pixel-to-nm calibration
calibration_nm_per_pixel = 0.017872741 * 1000  # Convert µm to nm

# Compute consistent colorbars for each component across all fractions
component_ranges = {}
for comp in components:
    all_preds_comp = [all_predictions[frac][comp] for frac in selected_fracs]
    valid_mask = mask > 0.5
    all_valid_preds_comp = np.concatenate([pred[valid_mask] for pred in all_preds_comp])
    vmin_comp = np.nanmin(all_valid_preds_comp)
    vmax_comp = np.nanmax(all_valid_preds_comp)
    component_ranges[comp] = (vmin_comp, vmax_comp)

# Plot in grid: rows = components, columns = sampling fractions
for row, component in enumerate(components):
    vmin_c, vmax_c = component_ranges[component]
    
    for col, frac in enumerate(selected_fracs):
        ax = axes[row, col]
        
        # Get prediction
        pred = all_predictions[frac][component]
        
        # Show prediction map with consistent colorbar range for this component
        im = ax.imshow(pred, cmap='RdBu', origin='lower', vmin=vmin_c, vmax=vmax_c, alpha=0.9)
        
        # Overlay training points as black dots
        sparse_mask_np = sparse_masks[frac].cpu().numpy().reshape(H, W)
        y_pts, x_pts = np.where(sparse_mask_np > 0.5)
        ax.scatter(x_pts, y_pts, c='black', s=4, alpha=0.4, edgecolors='white', 
                  linewidths=0.2, label='Training points')
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax, location='bottom', pad=0.06, shrink=0.85)
        cbar.set_label(component, fontsize=14, fontweight='bold')
        cbar.ax.tick_params(labelsize=12)
        
        # Title with metrics
        n_train = int(sparse_mask_np.sum())
        r2 = all_metrics[frac][component]['R²']
        rmse = all_metrics[frac][component]['RMSE']
        
        if col == 0:
            # Add component label on left
            ax.text(-0.35, 0.5, component, transform=ax.transAxes, 
                   fontsize=14, fontweight='bold', va='center', ha='right',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        ax.set_title(f'{frac*100:.0f}% ({n_train:,} pts)\nR²={r2:.3f}, RMSE={rmse:.2e}', 
                    fontweight='bold', fontsize=12)
        ax.set_xticks([])
        ax.set_yticks([])
        
        # Add scale bar (in nm, no label)
        scale_length_px = W // 8
        bar_x_start = W - scale_length_px - 20
        bar_x_end = W - 20
        bar_y = H - 20
        ax.plot([bar_x_start, bar_x_end], [bar_y, bar_y], 'k-', linewidth=8)

plt.tight_layout()
plt.savefig(out_dir / "10_combined_predictions_1_5_10pct.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Combined predictions (e_xx, e_yy, e_xy at 1%, 5%, 10%) saved")
print(f"  Calibration: {calibration_nm_per_pixel:.2f} nm/pixel")
print(f"\n  Component ranges (consistent colorbars):")
for comp, (vmin, vmax) in component_ranges.items():
    print(f"    {comp:6s}: {vmin:.4e} to {vmax:.4e}")

In [ ]:
# Ground truth with sampling overlays
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Ground Truth with Training Point Distribution', fontsize=14, fontweight='bold')

# Pixel-to-nm calibration
calibration_nm_per_pixel = 0.017872741 * 1000  # Convert µm to nm

for idx, frac in enumerate([0.01, 0.05, 0.10, 0.25, 0.50, 0.75]):
    ax = axes.flat[idx]
    
    # Show ground truth with consistent colorbar
    valid_gt = e_xx[mask > 0.5]
    vmin, vmax = np.nanmin(valid_gt), np.nanmax(valid_gt)
    im = ax.imshow(e_xx, cmap='RdBu', origin='lower', vmin=vmin, vmax=vmax, alpha=0.85)
    
    # Overlay training points
    sparse_mask_np = sparse_masks[frac].cpu().numpy().reshape(H, W)
    y_pts, x_pts = np.where(sparse_mask_np > 0.5)
    ax.scatter(x_pts, y_pts, c='black', s=5, alpha=0.5, edgecolors='white', linewidths=0.3, label='Training points')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, label='e_xx (ground truth)', location='bottom', pad=0.1)
    cbar.set_label('e_xx', fontsize=12, fontweight='bold')
    
    n_train = int(sparse_mask_np.sum())
    n_valid = int(mask.sum())
    ax.set_title(f'{frac*100:.0f}% Sampling\n({n_train:,} of {n_valid:,} points)', 
                 fontweight='bold', fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc='upper left', fontsize=14)
    
    # Add scale bar at bottom right (in nm, no label)
    scale_length_px = W // 8  # 1/8 of width in pixels
    bar_x_start = W - scale_length_px - 20
    bar_x_end = W - 20
    bar_y = H - 20
    ax.plot([bar_x_start, bar_x_end], [bar_y, bar_y], 'k-', linewidth=8)

plt.tight_layout()
plt.savefig(out_dir / "08_ground_truth_with_sampling_overlay.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Ground truth with sampling overlay saved to: {out_dir / '08_ground_truth_with_sampling_overlay.png'}")
print(f"  Calibration: {calibration_nm_per_pixel:.2f} nm/pixel")


In [ ]:
# Error maps with sampling overlays - showing where model struggles
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Prediction Errors with Training Point Distribution (e_xx)', fontsize=14, fontweight='bold')

for idx, frac in enumerate([0.01, 0.05, 0.10, 0.25, 0.50, 0.75]):
    ax = axes.flat[idx]
    
    # Compute absolute error
    e_xx_pred = all_predictions[frac]['e_xx']
    error_map = np.abs(e_xx_pred - e_xx)
    
    # Mask invalid regions
    error_display = np.copy(error_map)
    error_display[mask < 0.5] = 0.0
    
    # Use log scale for better visualization
    error_log = np.log10(error_display + 1e-10)
    vmin, vmax = -8, 0
    im = ax.imshow(error_log, cmap='hot', origin='lower', vmin=vmin, vmax=vmax)
    
    # Overlay training points
    sparse_mask_np = sparse_masks[frac].cpu().numpy().reshape(H, W)
    y_pts, x_pts = np.where(sparse_mask_np > 0.5)
    ax.scatter(x_pts, y_pts, c='blue', s=8, alpha=0.5, edgecolors='black', label='Training points')
    
    cbar = plt.colorbar(im, ax=ax, location='bottom', pad=0.06, shrink=0.85)
    cbar.set_label('log\u2081\u2080(|error|)', fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    n_train = int(sparse_mask_np.sum())
    mae = all_metrics[frac]['e_xx']['MAE']
    ax.set_title(f'{frac*100:.0f}% Sampling ({n_train:,} pts)\nMAE={mae:.3e}', fontweight='bold', fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc='upper left', fontsize=12)

plt.tight_layout()
plt.savefig(out_dir / "09_error_maps_with_sampling_overlay.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Error maps with sampling overlay saved to: {out_dir / '09_error_maps_with_sampling_overlay.png'}")


In [ ]:
# 4. Metrics vs. Sampling Fraction (comprehensive comparison)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Performance Metrics vs. Sampling Fraction', fontsize=14, fontweight='bold')

fracs_pct = [f*100 for f in config["sampling_fractions"]]

# R² score
ax = axes[0, 0]
for component in ['e_xx', 'e_yy', 'e_xy', 'theta']:
    r2_values = [all_metrics[f][component]['R²'] for f in config["sampling_fractions"]]
    ax.plot(fracs_pct, r2_values, marker='o', linewidth=2.5, markersize=8, label=component)
ax.set_xscale('log')
ax.set_xlabel('Sampling Fraction (%)', fontsize=11)
ax.set_ylabel('R² Score', fontsize=11)
ax.set_title('Model Fidelity (R²)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1.05])

# RMSE
ax = axes[0, 1]
for component in ['e_xx', 'e_yy', 'e_xy', 'theta']:
    rmse_values = [all_metrics[f][component]['RMSE'] for f in config["sampling_fractions"]]
    ax.semilogy(fracs_pct, rmse_values, marker='s', linewidth=2.5, markersize=8, label=component)
ax.set_xscale('log')
ax.set_xlabel('Sampling Fraction (%)', fontsize=11)
ax.set_ylabel('RMSE (log scale)', fontsize=11)
ax.set_title('Prediction Error (RMSE)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

# MAE
ax = axes[0, 2]
for component in ['e_xx', 'e_yy', 'e_xy', 'theta']:
    mae_values = [all_metrics[f][component]['MAE'] for f in config["sampling_fractions"]]
    ax.semilogy(fracs_pct, mae_values, marker='^', linewidth=2.5, markersize=8, label=component)
ax.set_xscale('log')
ax.set_xlabel('Sampling Fraction (%)', fontsize=11)
ax.set_ylabel('MAE (log scale)', fontsize=11)
ax.set_title('Mean Absolute Error (MAE)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

# Training time – show ms/epoch (removes epoch-count artefact at 25% & 75%)
ax = axes[1, 0]
times       = [training_times[f]                      for f in config["sampling_fractions"]]
epochs_run  = [len(training_histories[f]['epoch'])    for f in config["sampling_fractions"]]
ms_per_ep   = [1000 * t / e for t, e in zip(times, epochs_run)]
ax.plot(fracs_pct, ms_per_ep, marker='D', linewidth=2.5, markersize=8, color='purple')
for xv, yv, ep in zip(fracs_pct, ms_per_ep, epochs_run):
    if ep >= config['epochs']:
        ax.annotate(f'{ep:,} ep', xy=(xv, yv), xytext=(0, 10),
                    textcoords='offset points', fontsize=9, ha='center', color='#c0392b')
ax.set_xscale('log')
ax.set_xlabel('Sampling Fraction (%)', fontsize=11)
ax.set_ylabel('Time per Epoch (ms / epoch)', fontsize=11)
ax.set_title('Computational Cost per Epoch', fontweight='bold')
ax.grid(True, alpha=0.3)

# Convergence speed (log10 of final loss / initial loss)
ax = axes[1, 1]
convergence = []
for f in config["sampling_fractions"]:
    hist = training_histories[f]
    conv = np.log10(hist['loss_total'][-1] / hist['loss_total'][0])
    convergence.append(conv)
ax.bar(range(len(fracs_pct)), convergence, color='teal', alpha=0.7, width=0.6)
ax.set_xticks(range(len(fracs_pct)))
ax.set_xticklabels([f'{f:.0f}%' for f in fracs_pct], rotation=45)
ax.set_ylabel('log10(Final/Initial Loss)', fontsize=11)
ax.set_title('Convergence Factor', fontweight='bold')
ax.axhline(0, color='k', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3, axis='y')

# Data efficiency (RMSE gain per 1% additional data)
ax = axes[1, 2]
rmse_ref = all_metrics[config["sampling_fractions"][-1]]['e_xx']['RMSE']  # 100% reference (if available)
rmse_e_xx = [all_metrics[f]['e_xx']['RMSE'] for f in config["sampling_fractions"]]
efficiency = [rmse_e_xx[i] / (fracs_pct[i] / 100 + 1e-6) for i in range(len(fracs_pct))]
ax.loglog(fracs_pct, efficiency, marker='P', linewidth=2.5, markersize=8, color='orange')
ax.set_xlabel('Sampling Fraction (%)', fontsize=11)
ax.set_ylabel('RMSE / Fraction (log-log)', fontsize=11)
ax.set_title('Data Efficiency (e_xx)', fontweight='bold')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig(out_dir / "05_metrics_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Comprehensive metrics comparison saved")

In [ ]:
# 5. Box plots of errors across all fractions
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Error Distribution Across Sampling Fractions', fontsize=14, fontweight='bold')

for idx, component in enumerate(['e_xx', 'e_yy', 'e_xy', 'theta']):
    ax = axes.flat[idx]
    
    error_data = []
    labels_plot = []
    for frac in config["sampling_fractions"]:
        pred = all_predictions[frac][component]
        true = [e_xx, e_yy, e_xy, theta][idx]
        errors = np.abs(pred[mask > 0.5] - true[mask > 0.5])
        error_data.append(errors)
        labels_plot.append(f'{frac*100:.0f}%')
    
    try:                                 # [COLAB] matplotlib >= 3.9 renamed labels -> tick_labels
        bp = ax.boxplot(error_data, tick_labels=labels_plot, patch_artist=True)
    except TypeError:
        bp = ax.boxplot(error_data, labels=labels_plot, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    
    ax.set_ylabel('Absolute Error', fontsize=11)
    ax.set_xlabel('Sampling Fraction', fontsize=11)
    ax.set_title(f'{component}', fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_yscale('log')

plt.tight_layout()
plt.savefig(out_dir / "06_error_distributions.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Error distributions saved")

In [ ]:
# 6. Comprehensive summary table
print("\n" + "="*70)
print("FINAL RESULTS: Comprehensive Performance Summary")
print("="*70)

summary_rows = []
for frac in config["sampling_fractions"]:
    row = {
        'Sampling %': f'{frac*100:.1f}%',
        'Train Points': int((sparse_masks[frac].sum()).item()),
        '% of Grid': f'{100*int((sparse_masks[frac].sum()).item())/(H*W):.1f}%',
        'R² (e_xx)': f'{all_metrics[frac]["e_xx"]["R²"]:.4f}',
        'RMSE (e_xx)': f'{all_metrics[frac]["e_xx"]["RMSE"]:.4e}',
        'R² (avg)': f'{np.mean([all_metrics[frac][c]["R²"] for c in ["e_xx", "e_yy", "e_xy", "theta"]]):.4f}',
        'Time (s)': f'{training_times[frac]:.1f}',
        'Epochs': len(training_histories[frac]['epoch']),
    }
    summary_rows.append(row)

summary_table = pd.DataFrame(summary_rows)
print(summary_table.to_string(index=False))

# Save summary to CSV
csv_out = out_dir / "results_summary.csv"
summary_table.to_csv(csv_out, index=False)
print(f"\n✓ Summary saved to: {csv_out}")

# Save detailed metrics to CSV
detailed_metrics_list = []
for frac in config["sampling_fractions"]:
    for component in ['e_xx', 'e_yy', 'e_xy', 'theta']:
        for metric_name, metric_val in all_metrics[frac][component].items():
            detailed_metrics_list.append({
                'Sampling %': frac * 100,
                'Component': component,
                'Metric': metric_name,
                'Value': metric_val,
            })

detailed_df = pd.DataFrame(detailed_metrics_list)
detailed_csv = out_dir / "detailed_metrics.csv"
detailed_df.to_csv(detailed_csv, index=False)
print(f"✓ Detailed metrics saved to: {detailed_csv}")

print("\n" + "="*70)
print("KEY INSIGHTS")
print("="*70)

# Best performance
best_r2_frac = max(config["sampling_fractions"], key=lambda f: all_metrics[f]['e_xx']['R²'])
print(f"  ✓ Best e_xx R² achieved at: {best_r2_frac*100:.1f}% sampling (R²={all_metrics[best_r2_frac]['e_xx']['R²']:.4f})")

# Data efficiency sweet spot
print(f"  ✓ Recommendation for high accuracy (~0.90 R²):")
for frac in config["sampling_fractions"]:
    if all_metrics[frac]['e_xx']['R²'] > 0.90:
        print(f"      Use ≥{frac*100:.0f}% data (RMSE={all_metrics[frac]['e_xx']['RMSE']:.4e})")
        break

print(f"  ✓ Computational efficiency: Training times range from {min(training_times.values()):.1f}s to {max(training_times.values()):.1f}s")
print(f"  ✓ Total training time: {total_time:.1f}s")
print("\n" + "="*70)

In [ ]:
# ============================================================================
# SECTION 8b: Physics-consistency diagnostics (physical units)
#   R_equil  [GPa/um]   elastic-equilibrium residual
#   R_compat [1/um^2]   Saint-Venant compatibility residual
# Evaluated on the dense reconstructed grid with central finite differences.
# ============================================================================

physics_diag = {}
print("\n" + "="*70)
print("PHYSICS-CONSISTENCY DIAGNOSTICS (PINN reconstructions)")
print("="*70)
print(f"{'frac':>6} | {'R_equil [GPa/um]':>18} | {'R_compat [1/um^2]':>18}")
print("-"*52)
for frac in config["sampling_fractions"]:
    p = all_predictions[frac]
    R_eq, R_co = grid_physics_diagnostics(p["e_xx"], p["e_yy"], p["e_xy"], mask, config)
    physics_diag[frac] = {"R_equil": R_eq, "R_compat": R_co}
    print(f"{frac*100:5.0f}% | {R_eq:18.4e} | {R_co:18.4e}")

pd.DataFrame([
    {"Sampling %": f"{f*100:.0f}%", "R_equil (GPa/um)": physics_diag[f]["R_equil"],
     "R_compat (1/um^2)": physics_diag[f]["R_compat"]}
    for f in config["sampling_fractions"]
]).to_csv(out_dir / "physics_diagnostics.csv", index=False)
print(f"\n✓ Saved {out_dir / 'physics_diagnostics.csv'}")


In [ ]:
# ============================================================================
# SECTION 8c: Ablation -- data-only SIREN (lambda_phys = 0)
# Identical architecture, schedule and data; physics weight set to zero.
# Isolates the value of the PDE prior from network capacity.
# ============================================================================

ablation_models      = {}
ablation_predictions = {}
ablation_metrics     = {}
ablation_diag        = {}

print("\n" + "="*70)
print("ABLATION: data-only SIREN baseline (lambda_phys = 0)")
print("="*70)
for frac in config["sampling_fractions"]:
    print(f"\n▶ Ablation on {frac*100:.0f}% data ...")
    m0, _ = train_pinn_sparse(frac, sparse_masks[frac], config, device, lambda_phys_scale=0.0)
    preds0, mets0 = evaluate_model(m0, x_flat, y_flat, e_xx, e_yy, e_xy, theta,
                                   mask, H, W, device)
    R_eq0, R_co0 = grid_physics_diagnostics(preds0["e_xx"], preds0["e_yy"], preds0["e_xy"],
                                            mask, config)
    ablation_models[frac]      = m0
    ablation_predictions[frac] = preds0
    ablation_metrics[frac]     = mets0
    ablation_diag[frac]        = {"R_equil": R_eq0, "R_compat": R_co0}

# Comparison table: physics residual reduction and MAE change vs data-only SIREN
print("\n" + "="*78)
print("PINN vs data-only SIREN  (equal data, equal capacity, equal schedule)")
print("="*78)
print(f"{'frac':>6} | {'R_equil x':>10} | {'R_compat x':>11} | "
      f"{'MAE PINN':>10} | {'MAE SIREN':>10} | {'MAE redux':>9}")
print("-"*78)
ablation_rows = []
for frac in config["sampling_fractions"]:
    re_pinn = physics_diag[frac]["R_equil"];  re_abl = ablation_diag[frac]["R_equil"]
    rc_pinn = physics_diag[frac]["R_compat"]; rc_abl = ablation_diag[frac]["R_compat"]
    mae_pinn = all_metrics[frac]["e_xx"]["MAE"]
    mae_abl  = ablation_metrics[frac]["e_xx"]["MAE"]
    eq_ratio = re_abl / re_pinn if re_pinn > 0 else float("nan")
    co_ratio = rc_abl / rc_pinn if rc_pinn > 0 else float("nan")
    mae_redux = 100.0 * (mae_abl - mae_pinn) / mae_abl if mae_abl > 0 else float("nan")
    print(f"{frac*100:5.0f}% | {eq_ratio:10.2f} | {co_ratio:11.2f} | "
          f"{mae_pinn:10.4e} | {mae_abl:10.4e} | {mae_redux:8.1f}%")
    ablation_rows.append({
        "Sampling %": f"{frac*100:.0f}%",
        "R_equil PINN": re_pinn, "R_equil SIREN-only": re_abl, "R_equil ratio": eq_ratio,
        "R_compat PINN": rc_pinn, "R_compat SIREN-only": rc_abl, "R_compat ratio": co_ratio,
        "MAE PINN (e_xx)": mae_pinn, "MAE SIREN-only (e_xx)": mae_abl,
        "MAE reduction %": mae_redux,
    })

pd.DataFrame(ablation_rows).to_csv(out_dir / "ablation_summary.csv", index=False)
eq_ratios = [r["R_equil ratio"] for r in ablation_rows]
co_ratios = [r["R_compat ratio"] for r in ablation_rows]
print(f"\nEquilibrium-residual reduction: {min(eq_ratios):.1f}x - {max(eq_ratios):.1f}x")
print(f"Compatibility-residual reduction: {min(co_ratios):.1f}x - {max(co_ratios):.1f}x")
print(f"✓ Saved {out_dir / 'ablation_summary.csv'}")


In [ ]:
# ============================================================================
# SECTION 8d: Classical sparse-reconstruction baselines
#   CS : compressed sensing, L1 in a Daubechies wavelet basis solved by ISTA
#   GP : Gaussian-process regression with a Matern-3/2 kernel
# Both are evaluated on the same sampling masks as the PINN. The data-only SIREN
# (Section 8c ablation) is the third baseline. Equilibrium / compatibility
# residuals use the SAME finite-difference operator as the PINN diagnostics.
# ============================================================================
import pywt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel

def cs_reconstruct(channel_map, samp_mask_2d, n_iter=300, lam=None, wavelet="db4"):
    """Scattered-sample -> full image via ISTA with soft-thresholding in a 2D
    wavelet basis. M is a pixel-selection operator so ||M^T M|| = 1 (step = 1)."""
    M = samp_mask_2d > 0.5
    y = np.where(M, channel_map, 0.0).astype(np.float64)
    x = np.full_like(y, channel_map[M].mean())
    level = pywt.dwtn_max_level(channel_map.shape, wavelet)
    level = max(1, min(level, 4))
    coeffs0 = pywt.wavedec2(y, wavelet, level=level)
    arr0, slices = pywt.coeffs_to_array(coeffs0)
    if lam is None:
        lam = 0.02 * np.abs(arr0).max()
    for _ in range(n_iter):
        x[M] = channel_map[M]                          # enforce data consistency
        arr, sl = pywt.coeffs_to_array(pywt.wavedec2(x, wavelet, level=level))
        arr = np.sign(arr) * np.maximum(np.abs(arr) - lam, 0.0)  # soft threshold
        x = pywt.waverec2(pywt.array_to_coeffs(arr, sl, output_format="wavedec2"),
                          wavelet)[:channel_map.shape[0], :channel_map.shape[1]]
    x[M] = channel_map[M]
    return x

def gp_reconstruct(channel_map, samp_flat_idx, n_cap=1500):
    """GP regression (Matern-3/2) over normalised coordinates; train points
    capped at n_cap for tractability, prediction over the full grid in batches."""
    xs = x_flat.cpu().numpy(); ys = y_flat.cpu().numpy()
    vals = channel_map.flatten()
    idx = np.asarray(samp_flat_idx)
    if len(idx) > n_cap:
        idx = np.random.RandomState(SEED).choice(idx, n_cap, replace=False)
    Xtr = np.stack([xs[idx], ys[idx]], axis=1); ytr = vals[idx]
    ymu = ytr.mean()
    kernel = (ConstantKernel(1.0) * Matern(length_scale=0.1, nu=1.5)
              + WhiteKernel(noise_level=1e-3))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=False, alpha=1e-6)
    gp.fit(Xtr, ytr - ymu)
    Xall = np.stack([xs, ys], axis=1)
    pred = np.empty(Xall.shape[0])
    for s in range(0, Xall.shape[0], 8000):
        pred[s:s+8000] = gp.predict(Xall[s:s+8000])
    return (pred + ymu).reshape(channel_map.shape)

def _metrics(pred, true, m):
    pv, tv = pred[m], true[m]
    rmse = float(np.sqrt(np.mean((pv - tv) ** 2)))
    mae  = float(np.mean(np.abs(pv - tv)))
    ss_res = np.sum((tv - pv) ** 2); ss_tot = np.sum((tv - tv.mean()) ** 2)
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else 0.0
    return rmse, mae, r2

m_valid = mask > 0.5
baseline_rows = []
baseline_store = {}
print("\n" + "="*70)
print("BASELINES: compressed sensing (CS) and Gaussian process (GP)")
print("="*70)
for frac in config["sampling_fractions"]:
    sp2d = sparse_masks[frac].cpu().numpy().reshape(H, W)
    sidx = sampled_indices[frac]
    chans = {}
    for nm, cmap in [("e_xx", e_xx), ("e_yy", e_yy), ("e_xy", e_xy)]:
        chans[nm] = {
            "cs": cs_reconstruct(cmap, sp2d),
            "gp": gp_reconstruct(cmap, sidx),
        }
    baseline_store[frac] = chans
    for method in ["cs", "gp"]:
        rmse, mae, r2 = _metrics(chans["e_xx"][method], e_xx, m_valid)
        R_eq, R_co = grid_physics_diagnostics(
            chans["e_xx"][method], chans["e_yy"][method], chans["e_xy"][method], mask, config)
        baseline_rows.append({
            "Sampling %": f"{frac*100:.0f}%", "method": method.upper(),
            "RMSE (e_xx)": rmse, "MAE (e_xx)": mae, "R2 (e_xx)": r2,
            "R_equil (GPa/um)": R_eq, "R_compat (1/um^2)": R_co})
    # SIREN-only (ablation) row for the same fraction
    rmse, mae, r2 = (ablation_metrics[frac]["e_xx"]["RMSE"],
                     ablation_metrics[frac]["e_xx"]["MAE"],
                     ablation_metrics[frac]["e_xx"]["R²"])
    baseline_rows.append({
        "Sampling %": f"{frac*100:.0f}%", "method": "SIREN-only",
        "RMSE (e_xx)": rmse, "MAE (e_xx)": mae, "R2 (e_xx)": r2,
        "R_equil (GPa/um)": ablation_diag[frac]["R_equil"],
        "R_compat (1/um^2)": ablation_diag[frac]["R_compat"]})
    # PINN row
    baseline_rows.append({
        "Sampling %": f"{frac*100:.0f}%", "method": "PINN",
        "RMSE (e_xx)": all_metrics[frac]["e_xx"]["RMSE"],
        "MAE (e_xx)": all_metrics[frac]["e_xx"]["MAE"],
        "R2 (e_xx)": all_metrics[frac]["e_xx"]["R²"],
        "R_equil (GPa/um)": physics_diag[frac]["R_equil"],
        "R_compat (1/um^2)": physics_diag[frac]["R_compat"]})
    print(f"  {frac*100:5.0f}% done")

baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv(out_dir / "baselines_summary.csv", index=False)
print(f"\n✓ Saved {out_dir / 'baselines_summary.csv'}")

# Headline comparison at 10% sampling (MAE on e_xx, relative to PINN)
f10 = 0.10
sub = baseline_df[baseline_df["Sampling %"] == "10%"].set_index("method")
mae_pinn = sub.loc["PINN", "MAE (e_xx)"]
print(f"\nAt 10% sampling -- MAE(e_xx) reduction of PINN vs baseline:")
for meth in ["CS", "GP", "SIREN-only"]:
    mae_b = sub.loc[meth, "MAE (e_xx)"]
    print(f"  vs {meth:11s}: {100*(mae_b-mae_pinn)/mae_b:5.1f}%   "
          f"(R_equil ratio {sub.loc[meth,'R_equil (GPa/um)']/sub.loc['PINN','R_equil (GPa/um)']:.1f}x)")


In [ ]:
# ============================================================================
# SECTION 10: Bayesian Epistemic Uncertainty (MC Dropout + MFVI)
# ============================================================================

# --- MC Dropout SIREN ---
class StrainPINN_MCDropout(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=6, omega_0=1.0, drop_p=0.05):  # [FIX] was 0.1
        super().__init__()
        self.drop = nn.Dropout(p=drop_p)
        self.first = SirenLayer(2, hidden_dim, is_first=True, omega_0=omega_0)
        self.hidden_layers = nn.ModuleList([
            SirenLayer(hidden_dim, hidden_dim, is_first=False, omega_0=omega_0)
            for _ in range(num_layers - 2)
        ])
        self.skip_proj = nn.Linear(hidden_dim, hidden_dim)
        self.final = nn.Linear(hidden_dim, 4)
        with torch.no_grad():
            self.final.weight.uniform_(-np.sqrt(6 / hidden_dim), np.sqrt(6 / hidden_dim))

    def forward(self, x):
        x_h = self.first(x)                 # [FIX] no dropout on the first sine layer
        for i, layer in enumerate(self.hidden_layers):
            if i % 2 == 0 and i > 0:
                x_h = layer(x_h) + self.skip_proj(x_h)
            else:
                x_h = layer(x_h)
            x_h = self.drop(x_h)
        return self.final(x_h)

    def predict_uncertainty(self, x, T=50):
        self.train()  # keep dropout active
        with torch.no_grad():
            preds = torch.stack([self(x) for _ in range(T)], dim=0)
        return preds.mean(0), preds.std(0)


# --- Mean-Field Variational Inference SIREN ---
class BayesLinear(nn.Module):
    def __init__(self, in_features, out_features, prior_std=0.1):
        super().__init__()
        self.w_mu  = nn.Parameter(torch.zeros(out_features, in_features))
        self.w_rho = nn.Parameter(torch.full((out_features, in_features), -9.0))  # [FIX] was -5: noise ~ weight scale
        self.b_mu  = nn.Parameter(torch.zeros(out_features))
        self.b_rho = nn.Parameter(torch.full((out_features,), -9.0))               # [FIX] was -5
        self.prior_var = prior_std ** 2
        nn.init.kaiming_uniform_(self.w_mu, a=np.sqrt(5))

    @property
    def w_sig(self): return torch.log1p(torch.exp(self.w_rho))
    @property
    def b_sig(self): return torch.log1p(torch.exp(self.b_rho))

    def forward(self, x):
        if self.training:
            w = self.w_mu + self.w_sig * torch.randn_like(self.w_mu)
            b = self.b_mu + self.b_sig * torch.randn_like(self.b_mu)
        else:
            w, b = self.w_mu, self.b_mu
        return nn.functional.linear(x, w, b)

    def kl(self):
        pv = self.prior_var
        kl_w = 0.5 * ((self.w_sig**2 + self.w_mu**2) / pv - 1.0
                      - torch.log(self.w_sig**2 / pv))
        kl_b = 0.5 * ((self.b_sig**2 + self.b_mu**2) / pv - 1.0
                      - torch.log(self.b_sig**2 / pv))
        return kl_w.sum() + kl_b.sum()


class SirenBayes(nn.Module):
    def __init__(self, in_f, out_f, is_first=False, omega_0=1.0):
        super().__init__()
        self.omega_0 = omega_0
        self.linear = BayesLinear(in_f, out_f,
                                  prior_std=(1.0 / in_f if is_first else float(np.sqrt(6 / in_f)) / omega_0))  # [FIX] prior matched to SIREN weight scale
        with torch.no_grad():
            if is_first:
                self.linear.w_mu.uniform_(-1 / in_f, 1 / in_f)
            else:
                bound = np.sqrt(6 / in_f) / omega_0
                self.linear.w_mu.uniform_(-bound, bound)
    def forward(self, x): return torch.sin(self.omega_0 * self.linear(x))
    def kl(self):         return self.linear.kl()


class StrainPINN_MFVI(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=6, omega_0=1.0):
        super().__init__()
        self._omega_0 = omega_0
        self.first = SirenBayes(2, hidden_dim, is_first=True, omega_0=omega_0)
        self.hidden_layers = nn.ModuleList([
            SirenBayes(hidden_dim, hidden_dim, omega_0=omega_0)
            for _ in range(num_layers - 2)
        ])
        self.skip_proj = BayesLinear(hidden_dim, hidden_dim,
                                     prior_std=float(np.sqrt(6 / hidden_dim)) / omega_0)  # [FIX] matched prior
        self.final     = BayesLinear(hidden_dim, 4,
                                     prior_std=float(np.sqrt(6 / hidden_dim)))            # [FIX] matched prior

    def forward(self, x):
        x_h = self.first(x)
        for i, layer in enumerate(self.hidden_layers):
            if i % 2 == 0 and i > 0:
                x_h = layer(x_h) + torch.sin(self._omega_0 * self.skip_proj(x_h))
            else:
                x_h = layer(x_h)
        return self.final(x_h)

    def kl_total(self):
        kl = self.first.kl() + self.skip_proj.kl() + self.final.kl()
        for l in self.hidden_layers: kl += l.kl()
        return kl

    def predict_uncertainty(self, x, T=50):
        self.train()
        with torch.no_grad():
            preds = torch.stack([self(x) for _ in range(T)], dim=0)
        return preds.mean(0), preds.std(0)


# --- Shared Bayesian Training Loop ---
def train_bayes_model(model, sparse_mask, n_epochs=3000, lr=1e-3, kl_scale=1e-6):
    ema_state = {}                      # [FIX] running scale of the physics residuals
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=500, gamma=0.95)
    for epoch in trange(n_epochs, desc=model.__class__.__name__, leave=False):
        model.train()
        opt.zero_grad()
        lam = 1.0 - np.exp(-epoch / config["ramp_tau"])   # [FIX] ramp physics like the deterministic loop
        total, ld, lp = pinn_loss_with_physics(
            model, x_flat, y_flat,
            e_xx_scaled, e_yy_scaled, e_xy_scaled, theta_scaled,
            sparse_mask, H, W, device, config, ema_state, lam
        )
        dev = next(model.parameters()).device
        kl = model.kl_total() * kl_scale if hasattr(model, 'kl_total') else torch.zeros(1, device=dev).squeeze()
        loss = total + kl
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()


# --- Train at 10% sampling ---
TARGET_FRAC = 0.10
T_PASSES    = 150
sparse_10   = sparse_masks[TARGET_FRAC]

print('Training MC-Dropout PINN at 10% sampling ...')
mcd_model = StrainPINN_MCDropout(
    hidden_dim=config['hidden_dim'], num_layers=config['num_layers'],
    omega_0=config['omega_0']           # [FIX] was default 1.0
).to(device)
train_bayes_model(mcd_model, sparse_10, n_epochs=3000)

print('Training MFVI PINN at 10% sampling ...')
mfvi_model = StrainPINN_MFVI(
    hidden_dim=config['hidden_dim'], num_layers=config['num_layers'],
    omega_0=config['omega_0']           # [FIX] was default 1.0
).to(device)
train_bayes_model(mfvi_model, sparse_10, n_epochs=3000, kl_scale=1e-6)

print('Running stochastic inference ...')
inputs_all = torch.stack([x_flat, y_flat], dim=1)
mcd_mean_s,  mcd_std_s  = mcd_model.predict_uncertainty(inputs_all,  T=T_PASSES)
mfvi_mean_s, mfvi_std_s = mfvi_model.predict_uncertainty(inputs_all, T=T_PASSES)

def _to_map(t_scaled, name, H, W, mask_np):
    arr = unscale(t_scaled[:, 0], name).cpu().numpy().reshape(H, W)
    arr[~(mask_np > 0.5)] = np.nan
    return arr

def _std_map(t_std, H, W, mask_np):
    arr = (t_std[:, 0] * scalers['e_xx']['std']).cpu().numpy().reshape(H, W)  # [FIX] std in strain units, same as the mean maps
    arr[~(mask_np > 0.5)] = np.nan
    return arr

mcd_mean  = _to_map(mcd_mean_s,  'e_xx', H, W, mask)
mfvi_mean = _to_map(mfvi_mean_s, 'e_xx', H, W, mask)
mcd_std   = _std_map(mcd_std_s,  H, W, mask)
mfvi_std  = _std_map(mfvi_std_s, H, W, mask)
print('Uncertainty maps computed')

# --- Figure 6 ---
# --- Figure 6  (colorbar bottom, large fonts) ---
FS6     = 14
FS6_T   = 15
FS6_SUP = 15

valid_m = mask > 0.5
gt_plot = np.where(valid_m, e_xx, np.nan)
all_means = np.concatenate([mcd_mean[valid_m], mfvi_mean[valid_m], gt_plot[valid_m]])
vmin_m = float(np.nanpercentile(all_means, 2))
vmax_m = float(np.nanpercentile(all_means, 98))
all_stds = np.concatenate([mcd_std[valid_m], mfvi_std[valid_m]])
vmax_s   = float(np.nanpercentile(all_stds, 98))

def _cb6(im, ax, label):
    cb = plt.colorbar(im, ax=ax, location='bottom', pad=0.06, shrink=0.85)
    cb.set_label(label, fontsize=FS6)
    cb.ax.tick_params(labelsize=FS6 - 2)
    return cb

fig, axes = plt.subplots(2, 3, figsize=(15, 12))

# Row 0: GT | MCD mean | MCD std
im = axes[0,0].imshow(gt_plot,   cmap='RdBu',   origin='lower', vmin=vmin_m, vmax=vmax_m)
axes[0,0].set_title('Ground Truth $\\varepsilon_{xx}$', fontsize=FS6_T, fontweight='bold')
_cb6(im, axes[0,0], '$\\varepsilon_{xx}$')

im = axes[0,1].imshow(mcd_mean,  cmap='RdBu',   origin='lower', vmin=vmin_m, vmax=vmax_m)
axes[0,1].set_title('MC Dropout - Mean', fontsize=FS6_T, fontweight='bold')
_cb6(im, axes[0,1], '$\\varepsilon_{xx}$')

im = axes[0,2].imshow(mcd_std,   cmap='YlOrRd', origin='lower', vmin=0, vmax=vmax_s)
axes[0,2].set_title('MC Dropout - Std (Epistemic)', fontsize=FS6_T, fontweight='bold')
_cb6(im, axes[0,2], 'Uncertainty (std)')

# Row 1: sampling overlay | MFVI mean | MFVI std
sp_np = sparse_masks[TARGET_FRAC].cpu().numpy().reshape(H, W)
# Show GT underneath; overlay training pixels as 1-px black dots (avoids scatter clutter)
axes[1,0].imshow(gt_plot, cmap='RdBu', origin='lower', vmin=vmin_m, vmax=vmax_m, alpha=0.55)
rgba_pts = np.zeros((H, W, 4), dtype=np.float32)
rgba_pts[sp_np > 0.5] = [0.05, 0.05, 0.05, 0.85]
axes[1,0].imshow(rgba_pts, origin='lower', interpolation='nearest')
axes[1,0].set_title(f'Training Points (10%, n={int(sp_np.sum()):,})', fontsize=FS6_T, fontweight='bold')

im = axes[1,1].imshow(mfvi_mean, cmap='RdBu',   origin='lower', vmin=vmin_m, vmax=vmax_m)
axes[1,1].set_title('MFVI - Mean', fontsize=FS6_T, fontweight='bold')
_cb6(im, axes[1,1], '$\\varepsilon_{xx}$')

im = axes[1,2].imshow(mfvi_std,  cmap='YlOrRd', origin='lower', vmin=0, vmax=vmax_s)
axes[1,2].set_title('MFVI - Std (Epistemic)', fontsize=FS6_T, fontweight='bold')
_cb6(im, axes[1,2], 'Uncertainty (std)')

for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(
    f'Bayesian Epistemic Uncertainty -- $\\varepsilon_{{xx}}$ at 10% Sparse Data (T = {T_PASSES})',
    fontsize=FS6_SUP, fontweight='bold'
)
plt.tight_layout()
fig.savefig(out_dir / 'fig6_uq_exx.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {out_dir / "fig6_uq_exx.png"}')


In [ ]:
# ============================================================================
# PAPER FIGURE EXPORT
# Creates paper/Fig/ and saves all 7 publication-ready figures
# ============================================================================
import shutil

PDPI   = 300
FS     = 14   # base font size for labels / ticks
FS_T   = 15   # subplot title
FS_SUP = 16   # suptitle
paper_dir = Path('paper/Fig')
paper_dir.mkdir(parents=True, exist_ok=True)
print(f'Output: {paper_dir.resolve()}')

valid_m = mask > 0.5

def _cbar_bottom(im, ax, label):
    cb = plt.colorbar(im, ax=ax, location='bottom', pad=0.06, shrink=0.85)
    cb.set_label(label, fontsize=FS)
    cb.ax.tick_params(labelsize=FS - 2)
    return cb

# ---- fig1_raw_data.png  (colorbar bottom, large fonts) ----------------
fig, axes = plt.subplots(2, 3, figsize=(15, 12))
panels = [
    (e_xx,  '$\\varepsilon_{xx}$',   'RdBu'),
    (e_yy,  '$\\varepsilon_{yy}$',   'RdBu'),
    (e_xy,  '$\\varepsilon_{xy}$',   'RdBu'),
    (theta, '$\\theta$ (rotation)',  'RdBu'),
    (mask,  'Valid Mask',            'gray'),
    (error, 'Measurement Error',     'viridis'),
]
for ax, (arr, lbl, cmap) in zip(axes.flat, panels):
    disp = np.nan_to_num(arr.astype(float), nan=0.0, posinf=0.0, neginf=0.0)
    fv   = disp[np.isfinite(disp)]
    vmin, vmax = float(fv.min()), float(fv.max())
    if vmin == vmax: vmin, vmax = vmin - 1e-6, vmax + 1e-6
    im = ax.imshow(disp, cmap=cmap, origin='lower', vmin=vmin, vmax=vmax)
    _cbar_bottom(im, ax, lbl)
    ax.set_title(lbl, fontsize=FS_T, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
fig.savefig(paper_dir / 'fig1_raw_data.png', dpi=PDPI, bbox_inches='tight')
plt.close(fig); print('Saved: fig1_raw_data.png')

# ---- fig1b_sampling.png -----------------------------------------------
gt_bg = np.where(valid_m, e_xx, np.nan)
vg2, vg98 = np.nanpercentile(gt_bg[valid_m], [2, 98])
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, frac in zip(axes.flat, config['sampling_fractions']):
    sp = sparse_masks[frac].cpu().numpy().reshape(H, W)
    ax.imshow(gt_bg, cmap='RdBu', origin='lower', vmin=vg2, vmax=vg98, alpha=0.50)
    rgba_pts = np.zeros((H, W, 4), dtype=np.float32)
    rgba_pts[sp > 0.5] = [0.05, 0.05, 0.05, 0.80]
    ax.imshow(rgba_pts, origin='lower', interpolation='nearest')
    ax.set_title(f'{frac*100:.0f}%  ({int(sp.sum()):,} pts)', fontsize=FS_T, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
fig.savefig(paper_dir / 'fig1b_sampling.png', dpi=PDPI, bbox_inches='tight')
plt.close(fig); print('Saved: fig1b_sampling.png')

# ---- fig2_convergence.png  (larger fonts) -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
for frac in config['sampling_fractions']:
    h = training_histories[frac]
    ax.semilogy(h['epoch'], h['loss_total'], lw=2.5, label=f'{frac*100:.0f}%')
ax.set_xlabel('Epoch', fontsize=FS); ax.set_ylabel('Total Loss (log)', fontsize=FS)
ax.set_title('Convergence vs Sampling Fraction', fontsize=FS_T, fontweight='bold')
ax.tick_params(labelsize=FS - 1)
ax.legend(ncol=2, fontsize=FS - 1); ax.grid(True, alpha=0.3)
ax = axes[1]
h50 = training_histories[0.50]
ax.semilogy(h50['epoch'], h50['loss_data'],    lw=2.5, c='green', label='Data loss')
ax.semilogy(h50['epoch'], h50['loss_physics'], lw=2.5, c='red',   label='Physics loss')
ax.set_xlabel('Epoch', fontsize=FS); ax.set_ylabel('Loss (log)', fontsize=FS)
ax.set_title('Loss Breakdown at 50% Sampling', fontsize=FS_T, fontweight='bold')
ax.tick_params(labelsize=FS - 1)
ax.legend(fontsize=FS - 1); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(paper_dir / 'fig2_convergence.png', dpi=PDPI, bbox_inches='tight')
plt.close(fig); print('Saved: fig2_convergence.png')

# ---- fig3_pred_exx.png  (colorbar bottom, larger fonts) ---------------
all_p  = [all_predictions[f]['e_xx'] for f in config['sampling_fractions']]
concat = np.concatenate([p[valid_m] for p in all_p])
vp2, vp98 = np.nanpercentile(concat, [2, 98])
fig, axes = plt.subplots(2, 3, figsize=(16, 12))
for ax, frac in zip(axes.flat, config['sampling_fractions']):
    pred = all_predictions[frac]['e_xx']
    im   = ax.imshow(pred, cmap='RdBu', origin='lower', vmin=vp2, vmax=vp98)
    sp   = sparse_masks[frac].cpu().numpy().reshape(H, W)
    yp, xp = np.where(sp > 0.5)
    ax.scatter(xp, yp, c='limegreen', s=4, alpha=0.5, linewidths=0)
    _cbar_bottom(im, ax, '$\\varepsilon_{xx}$')
    r2   = all_metrics[frac]['e_xx']['R2'] if 'R2' in all_metrics[frac]['e_xx'] else all_metrics[frac]['e_xx'].get('R²', float('nan'))
    rmse = all_metrics[frac]['e_xx']['RMSE']
    ax.set_title(f'{frac*100:.0f}%  ({int(sp.sum()):,} pts)\n$R^2$={r2:.3f}  RMSE={rmse:.3f}',
                 fontsize=FS_T, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
fig.savefig(paper_dir / 'fig3_pred_exx.png', dpi=PDPI, bbox_inches='tight')
plt.close(fig); print('Saved: fig3_pred_exx.png')

# ---- fig4_error_heatmaps.png  (colorbar bottom, larger fonts) ---------
fig, axes = plt.subplots(2, 3, figsize=(15, 12))
for ax, frac in zip(axes.flat, config['sampling_fractions']):
    err = np.abs(all_predictions[frac]['e_xx'] - e_xx)
    err_m = np.where(valid_m, err, np.nan)
    vmax_e = np.nanpercentile(err_m[valid_m], 98)
    im = ax.imshow(err_m, cmap='hot', origin='lower', vmin=0, vmax=vmax_e)
    _cbar_bottom(im, ax, '|Error|')
    mae = all_metrics[frac]['e_xx']['MAE']
    ax.set_title(f'{frac*100:.0f}%  MAE={mae:.3f}', fontsize=FS_T, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
fig.savefig(paper_dir / 'fig4_error_heatmaps.png', dpi=PDPI, bbox_inches='tight')
plt.close(fig); print('Saved: fig4_error_heatmaps.png')

# ---- fig5_metrics.png -------------------------------------------------
fracs_pct = [f * 100 for f in config['sampling_fractions']]
comps  = ['e_xx', 'e_yy', 'e_xy', 'theta']
labels = ['$\\varepsilon_{xx}$', '$\\varepsilon_{yy}$',
          '$\\varepsilon_{xy}$', '$\\theta$']
mks    = ['o', 's', '^', 'D']
r2_key = 'R2' if 'R2' in all_metrics[config['sampling_fractions'][0]]['e_xx'] else 'R²'
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

ax = axes[0,0]
for c, lb, mk in zip(comps, labels, mks):
    ax.plot(fracs_pct, [all_metrics[f][c][r2_key] for f in config['sampling_fractions']],
            marker=mk, lw=2.5, ms=8, label=lb)
ax.set_xscale('log'); ax.axhline(0, ls='--', c='k', alpha=0.4)
ax.set_xlabel('Sampling (%)', fontsize=FS); ax.set_ylabel('$R^2$', fontsize=FS)
ax.set_title('$R^2$ Score', fontsize=FS_T, fontweight='bold')
ax.tick_params(labelsize=FS - 1); ax.legend(fontsize=FS - 1); ax.grid(True, alpha=0.3)

ax = axes[0,1]
for c, lb, mk in zip(comps, labels, mks):
    ax.semilogy(fracs_pct, [all_metrics[f][c]['RMSE'] for f in config['sampling_fractions']],
                marker=mk, lw=2.5, ms=8, label=lb)
ax.set_xscale('log'); ax.set_xlabel('Sampling (%)', fontsize=FS); ax.set_ylabel('RMSE', fontsize=FS)
ax.set_title('RMSE', fontsize=FS_T, fontweight='bold')
ax.tick_params(labelsize=FS - 1); ax.legend(fontsize=FS - 1); ax.grid(True, alpha=0.3)

ax = axes[0,2]
for c, lb, mk in zip(comps, labels, mks):
    ax.semilogy(fracs_pct, [all_metrics[f][c]['MAE'] for f in config['sampling_fractions']],
                marker=mk, lw=2.5, ms=8, label=lb)
ax.set_xscale('log'); ax.set_xlabel('Sampling (%)', fontsize=FS); ax.set_ylabel('MAE', fontsize=FS)
ax.set_title('MAE', fontsize=FS_T, fontweight='bold')
ax.tick_params(labelsize=FS - 1); ax.legend(fontsize=FS - 1); ax.grid(True, alpha=0.3)

ax = axes[1,0]
_times      = [training_times[f]                   for f in config['sampling_fractions']]
_ep_run     = [len(training_histories[f]['epoch'])  for f in config['sampling_fractions']]
_ms_per_ep  = [1000 * t / e for t, e in zip(_times, _ep_run)]
ax.plot(fracs_pct, _ms_per_ep, marker='D', lw=2.5, ms=8, color='purple')
for xv, yv, ep in zip(fracs_pct, _ms_per_ep, _ep_run):
    if ep >= config['epochs']:
        ax.annotate(f'{ep:,} ep', xy=(xv, yv), xytext=(0, 10),
                    textcoords='offset points', fontsize=FS-3, ha='center', color='#c0392b')
ax.set_xscale('log'); ax.set_xlabel('Sampling (%)', fontsize=FS); ax.set_ylabel('ms / epoch', fontsize=FS)
ax.set_title('Cost per Epoch', fontsize=FS_T, fontweight='bold')
ax.tick_params(labelsize=FS - 1); ax.grid(True, alpha=0.3)

ax = axes[1,1]
conv = [np.log10(training_histories[f]['loss_total'][-1] /
                  training_histories[f]['loss_total'][0])
        for f in config['sampling_fractions']]
ax.bar(range(len(fracs_pct)), conv, color='teal', alpha=0.7)
ax.set_xticks(range(len(fracs_pct)))
ax.set_xticklabels([f'{fp:.0f}%' for fp in fracs_pct], rotation=45, fontsize=FS - 1)
ax.set_ylabel('log(Final/Initial Loss)', fontsize=FS)
ax.set_title('Convergence Factor', fontsize=FS_T, fontweight='bold')
ax.tick_params(labelsize=FS - 1)
ax.axhline(0, c='k', ls='--', alpha=0.5); ax.grid(True, alpha=0.3, axis='y')

ax = axes[1,2]
rmse_exx = [all_metrics[f]['e_xx']['RMSE'] for f in config['sampling_fractions']]
ax.loglog(fracs_pct, rmse_exx, marker='P', lw=2.5, ms=9, color='orange')
ax.set_xlabel('Sampling (%)', fontsize=FS); ax.set_ylabel('RMSE ($\\varepsilon_{xx}$)', fontsize=FS)
ax.set_title('Data Efficiency', fontsize=FS_T, fontweight='bold')
ax.tick_params(labelsize=FS - 1); ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(paper_dir / 'fig5_metrics.png', dpi=PDPI, bbox_inches='tight')
plt.close(fig); print('Saved: fig5_metrics.png')

# ---- fig6_uq_exx.png --------------------------------------------------
src6 = out_dir / 'fig6_uq_exx.png'
if src6.exists():
    shutil.copy(src6, paper_dir / 'fig6_uq_exx.png')
    print('Saved: fig6_uq_exx.png  (copied from outputs/)')
else:
    print('WARNING: fig6_uq_exx.png missing -- run the Bayesian UQ cell first')

# ---- Checklist --------------------------------------------------------
print()
required = ['fig1_raw_data.png', 'fig1b_sampling.png', 'fig2_convergence.png',
            'fig3_pred_exx.png', 'fig4_error_heatmaps.png',
            'fig5_metrics.png',  'fig6_uq_exx.png']
print('=== Paper figure checklist ===')
for fn in required:
    exists = (paper_dir / fn).exists()
    print(f'  {"OK" if exists else "MISSING"}  {fn}')
print(f'\nAll figures in: {paper_dir.resolve()}')


## CONCLUSIONS & FUTURE DIRECTIONS

### What this notebook shows

1. **Data efficiency.** A SIREN coordinate network recovers the dominant strain structure of the
   PbGeSnSe₁.₅Te₁.₅ specimen from a small fraction of probe positions: R²(e_xx) ≈ 0.44 / 0.72 / 0.80 / 0.84 /
   0.85 / 0.86 at 1 / 5 / 10 / 25 / 50 / 75% sampling. Quality saturates by ≈25%; reconstructions at
   10–25% already capture the chevron strain bands. The remaining error is concentrated in
   pixel-scale speckle — largely measurement noise (~3% of the field variance) that no method can
   recover from sub-sampled data.
2. **Against classical baselines.** At 10% sampling the physics-informed model reduces MAE(e_xx) by
   ≈26% vs compressed sensing (wavelet ISTA) and ≈22% vs Gaussian-process regression (Section 8d).
3. **When physics helps — and when it does not.** Against an identical data-only SIREN (equal data,
   capacity, and schedule; Section 8c), the elasticity prior improves MAE(e_xx) at 1% sampling
   (≈10%), is roughly neutral at 5%, and *degrades* accuracy at ≥10% — while always reducing the
   physics residuals (equilibrium 1.5–1.8×, compatibility 2.2–3.9×). The prior is therefore a
   **regulariser for the extreme-sparsity regime**, not a universal accuracy booster. A plausible
   physical reason: homogeneous, isotropic plane-stress equilibrium is only approximate for a
   domain-structured, ferroelastic crystal, where the spontaneous (transformation) eigenstrain of
   the domains makes ∇·(C:ε_measured) ≠ 0, so enforcing it biases the reconstruction once the
   data alone suffice.
4. **Uncertainty quantification.** MC-dropout and mean-field-VI variants (Section 10) yield
   epistemic uncertainty maps at 10% sampling that highlight under-sampled and high-gradient
   regions — useful as an acquisition-steering signal.

### Practical implications for acquisition

If the quantity of interest is strain-band morphology rather than pixel-level speckle, **10–25% of
probe positions suffice**, i.e. a ≈4–10× reduction in electron dose, scan time, and data volume.
This is directly relevant to beam-sensitive materials and in-situ experiments; the uncertainty maps
suggest a route to adaptive (steered) acquisition.

### Limitations and caveats

- Physics residuals are formed on per-component standardised fields; unequal component stds slightly
  distort ν in the elastic operator. A cleaner formulation would build residuals on physical-unit
  strains.
- Normalisation statistics (mean/std) are computed from the full field — a mild information leak
  relative to a strict sparse-acquisition scenario.
- θ is *derived* from (e_xx, e_yy, e_xy) via an arctan2 expression, not an independently measured
  rotation; its reconstruction scores should be read accordingly.
- The GP baseline is capped at 1,500 training points for tractability, so comparisons at ≥25%
  sampling understate GP performance.
- One dataset and one mask realisation per fraction; repeated random masks would give error bars.

### Future directions

1. **Eigenstrain-aware physics**: jointly infer a domain-eigenstrain field so the equilibrium
   residual is exact for ferroelastic microstructures — this could extend the benefit of the prior
   beyond the 1% regime.
2. **Adaptive loss balancing** (GradNorm / NTK-based) in place of the fixed ramp + EMA
   normalisation used here.
3. **Aspect-corrected coordinates** (x ∈ [0, W/H]): a small verified gain (+0.005–0.01 R²) by making
   frequencies isotropic in pixel space.
4. **Noise-aware evaluation**: report R²/SSIM against a lightly denoised reference (e.g. median-
   filtered maps) alongside raw ground truth.
5. **Extensions**: multi-modal fusion (strain + DPC + vDFI with coupled physics), inverse problems
   (inferring E, ν or composition from strain), and steered acquisition driven by the UQ maps.

### References

- [SIREN: Implicit Neural Representations with Periodic Activation Functions](https://arxiv.org/abs/2006.09661)
- [Curriculum-Enhanced Adaptive Sampling for PINNs](https://www.mdpi.com/2227-7390/13/24/3996)
- [R-PINN: Recovery-type A-Posteriori Enhanced PINN](https://arxiv.org/html/2506.10243)
- [Failure-Informed Adaptive Sampling](https://epubs.siam.org/doi/10.1137/22M1527763)
